# StageBridge: Niche-Level Lung Adenocarcinoma Stage Classification

**Primary research notebook** — end-to-end entry point for the full EA-MIST pipeline, from raw data to publication figures.

## Pipeline Overview

| Part | Purpose | Key Output |
|------|---------|------------|
| **I. Setup** | Configure run, validate environment | Paths, GPU, DR backends |
| **II. Data Preprocessing** | Load snRNA-seq, Visium, WES; 4-method DR | Cohort tables, PCA/UMAP/t-SNE/PHATE embeddings |
| **III. Reference Mapping** | HLCA + LuCA atlas embedding | Cosine similarity profiles (13D + 15D) |
| **IV. Spatial Providers** | Tangram / TACCO / DestVI deconvolution | Cell-type compositions, provider QC |
| **V. EA-MIST Bags** | Lesion bag construction + niche/lesion-level DR | 56 lesion bags, 639K neighborhoods, multi-scale embeddings |
| **VI. Atlas Ablation** | 3×5 grouped ordinal benchmark | HPO results, best configs per fold |
| **VII. Results** | Metrics, confusion matrices, advanced comparisons | Radar/parallel coords, violins, ridge plots |
| **VIII. Transcriptomics** | Cell-type profiles, clustermaps, correlation | Dendrograms, effect sizes, cross-atlas structure |
| **IX. Figures & Summary** | Composite multi-panel + full inventory | 23+ publication figures (PNG + PDF) |

### Architecture

EA-MIST (Evolutionary Atlas-informed Multiple Instance Set Transformer) treats each lesion as a **bag of spatial neighborhoods**. Each neighborhood is tokenized (receiver cell, ring compositions, HLCA/LuCA similarities, L/R pathways, statistics), encoded by a local transformer, then aggregated by a set transformer with prototype bottleneck into lesion-level predictions.

### Dimensionality Reduction Methods

| Method | Type | Key property |
|--------|------|-------------|
| **PCA** | Linear | Explained variance % on axes; scree plots for intrinsic dimensionality |
| **UMAP** | Non-linear | Local + global topology; density contours and confidence ellipses |
| **t-SNE** | Non-linear | Crisp local clusters; adaptive perplexity |
| **PHATE** | Non-linear | Continuous trajectories; diffusion-based (falls back to UMAP if unavailable) |

### Evaluation

Grouped ordinal 3-class labels (early_like / intermediate_like / invasive_like) with donor-held-out 3-fold CV, 50-trial HPO, and ablation across 5 atlas configurations × 3 model families.

In [ ]:
# --- Part I: Configuration and Imports ---
import os, warnings
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import torch
from torch import nn
from IPython.display import Markdown, display

# Dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    warnings.warn("umap-learn not installed; UMAP panels will fall back to PCA.")

try:
    import phate
    HAS_PHATE = True
except ImportError:
    HAS_PHATE = False
    warnings.warn("phate not installed; PHATE panels will fall back to UMAP/PCA.")

from scipy.stats import gaussian_kde, spearmanr
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.patches import Ellipse, Patch, FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe

from stagebridge.notebook_api import (
    compose_config,
    clone_config,
    run_step,
    run_data_preprocessing_overview,
    build_dataset_preprocessing_table,
    run_reference,
    build_reference_summary_table,
    build_reference_evaluation_table,
    build_reference_label_table,
    run_spatial_provider_ladder,
    build_spatial_provider_metric_table,
    build_spatial_provider_agreement_table,
    load_run,
)
from stagebridge.viz.research_frontend import (
    configure_research_style,
    plot_multi_embedding_frontend,
    plot_reference_frontend,
)
from stagebridge.viz.advanced_plots import (
    plot_radar_chart,
    plot_parallel_coordinates,
    plot_correlation_matrix,
    plot_3d_embedding,
    plot_ridge_distributions,
)
from stagebridge.viz.eamist_figures import (
    save_method_overview_figure,
    save_embedding_diagnostics_figure,
    save_benchmark_comparison_figure,
    save_ablation_figure,
    save_prototype_interpretation_figure,
)
from stagebridge.data.luad_evo.stages import (
    CANONICAL_STAGE_ORDER, GROUPED_STAGE_ORDER, STAGE_TO_GROUP,
)

# EA-MIST model architecture imports
from stagebridge.context_model.lesion_set_transformer import EAMISTModel, EAMISTOutput
from stagebridge.context_model.prototype_bottleneck import (
    PrototypeBottleneck, PrototypeBottleneckOutput,
    prototype_diversity_loss, assignment_entropy_loss, prototype_orthogonality_loss,
)
from stagebridge.context_model.local_niche_encoder import (
    LocalNicheTokenizer, LocalNicheTransformerEncoder, LocalNicheEncoderOutput,
)
from stagebridge.context_model.set_encoder import SAB, ISAB, PMA
from stagebridge.context_model.evolution_branch import EvolutionBranch
from stagebridge.context_model.losses import (
    ordinal_stage_loss, displacement_regression_loss,
    transition_consistency_loss, lesion_subsampling_consistency_loss,
)
from stagebridge.context_model.communication_builder import (
    LUNG_LR_PRIORS, RECEIVER_PROGRAMS, CommunicationPrior,
    FAMILY_TO_PROGRAM,
)
from stagebridge.context_model.token_schema import (
    DEFAULT_TYPED_FEATURE_NAMES, default_typed_token_schema,
)
from stagebridge.pipelines.pretrain_local import LocalFeatureDims
from stagebridge.pipelines.train_lesion import build_model_family
from stagebridge.data.luad_evo.bag_dataset import LesionBagDataset, collate_lesion_bags
from stagebridge.utils.types import LesionBagBatch

# ── Publication-quality style ──────────────────────────────────────
configure_research_style()
# Override with tighter, publication-friendly settings
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,          # editable text in PDF
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})

# ── Color palettes ────────────────────────────────────────────────
STAGE_COLORS = {
    "Normal": "#00BA38", "AAH": "#F8766D", "AIS": "#619CFF",
    "MIA": "#E58700", "LUAD": "#A3A500",
}
GROUP_COLORS = {
    "early_like": "#4CAF50", "intermediate_like": "#FF9800", "invasive_like": "#F44336",
}
MODEL_COLORS = {"pooled": "#7570B3", "deep_sets": "#D95F02", "eamist": "#1B9E77"}

# Token type names and colors for local niche encoder visualization
TOKEN_TYPE_NAMES = [
    "Receiver", "Ring (x4)", "HLCA atlas", "LuCA atlas",
    "LR pathway", "Niche stats", "Atlas contrast"
]
TOKEN_TYPE_COLORS = [
    "#E41A1C", "#FF7F00", "#2166AC", "#B2182B",
    "#4DAF4A", "#984EA3", "#A65628"
]

# Prototype palette (K=16)
PROTO_CMAP = plt.cm.get_cmap("tab20", 16)

# LR family colors
LR_FAMILY_COLORS = {
    "inflammatory": "#E41A1C", "chemokine": "#377EB8", "tgfb": "#4DAF4A",
    "growth_factor": "#FF7F00", "notch": "#984EA3", "ecm": "#A65628",
    "vascular": "#F781BF", "immune_modulatory": "#999999", "developmental": "#66C2A5",
}

# ── Shared DR helper ──────────────────────────────────────────────
def compute_all_embeddings(X, n_subsample=5000, seed=42):
    """Compute PCA (+ variance%), UMAP, t-SNE, PHATE on feature matrix X.
    Returns dict of {method: (coords_2d, metadata_str)}."""
    rng = np.random.default_rng(seed)
    if X.shape[0] > n_subsample:
        idx = rng.choice(X.shape[0], n_subsample, replace=False)
        X = X[idx]
    else:
        idx = np.arange(X.shape[0])

    # PCA
    pca = PCA(n_components=min(3, X.shape[1]), random_state=seed)
    pca_coords = pca.fit_transform(X)
    var = pca.explained_variance_ratio_ * 100
    pca_label = f"PC1={var[0]:.1f}%, PC2={var[1]:.1f}%"
    cumvar = np.cumsum(var)
    n90 = int(np.searchsorted(cumvar, 90.0) + 1)

    result = {
        "PCA": (pca_coords[:, :2], pca_label),
        "pca_3d": pca_coords[:, :3] if pca_coords.shape[1] >= 3 else None,
        "pca_var": var,
        "pca_n90": n90,
    }

    # UMAP
    if HAS_UMAP:
        try:
            u = umap.UMAP(n_components=2, n_neighbors=min(30, X.shape[0]-1),
                          min_dist=0.3, random_state=seed)
            result["UMAP"] = (u.fit_transform(X), "")
        except Exception:
            result["UMAP"] = (pca_coords[:, :2], "(fallback PCA)")
    else:
        result["UMAP"] = (pca_coords[:, :2], "(fallback PCA)")

    # t-SNE
    try:
        perp = min(50.0, max(5.0, float(X.shape[0] - 1) / 3.0))
        tsne = TSNE(n_components=2, perplexity=perp, random_state=seed,
                    init="pca", learning_rate="auto")
        result["t-SNE"] = (tsne.fit_transform(X), f"perp={perp:.0f}")
    except Exception:
        result["t-SNE"] = (pca_coords[:, :2], "(fallback PCA)")

    # PHATE
    if HAS_PHATE:
        try:
            ph = phate.PHATE(n_components=2, random_state=seed, n_jobs=1, verbose=0)
            result["PHATE"] = (ph.fit_transform(X), "")
        except Exception:
            result["PHATE"] = result["UMAP"]
    else:
        result["PHATE"] = result["UMAP"]

    result["_idx"] = idx
    return result


def plot_four_embeddings(embeddings, labels, label_colors, title,
                         output_path=None, figsize=(22, 5.5), point_size=8):
    """Publication-quality 4-panel (PCA/UMAP/t-SNE/PHATE) figure."""
    methods = ["PCA", "UMAP", "t-SNE", "PHATE"]
    fig, axes = plt.subplots(1, 4, figsize=figsize)
    for ax, method in zip(axes, methods):
        coords, meta = embeddings[method]
        subtitle = f"{method}  {meta}" if meta else method
        for lab in dict.fromkeys(labels):  # preserve order, deduplicate
            mask = np.array(labels) == lab
            if not mask.any():
                continue
            ax.scatter(coords[mask, 0], coords[mask, 1], s=point_size, alpha=0.6,
                       color=label_colors.get(lab, "#999999"), label=lab,
                       linewidths=0.0, rasterized=True)
        ax.set_title(subtitle, fontsize=11, fontweight="bold")
        ax.set_xlabel(f"{method} 1" if method != "PCA" else "PC 1", fontsize=10)
        ax.set_ylabel(f"{method} 2" if method != "PCA" else "PC 2", fontsize=10)
        ax.tick_params(labelsize=8)
        ax.legend(frameon=True, fontsize=7, markerscale=1.5, edgecolor="gray",
                  fancybox=True, framealpha=0.9)
    fig.suptitle(title, fontsize=15, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    if output_path:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        fig.savefig(Path(output_path).with_suffix(".pdf"), bbox_inches="tight")
    return fig


def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    """Draw an n_std confidence ellipse on *ax*."""
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=(np.mean(x), np.mean(y)), width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(ell)


def load_eamist_checkpoint(checkpoint_path, cfg, device="cpu"):
    """Load a trained EAMISTModel from a checkpoint file.
    Returns (model, ckpt_dict) or (None, None) if loading fails."""
    try:
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        dims = LocalFeatureDims(**ckpt["dims"])
        model = build_model_family(
            ckpt["model_family"], dims, cfg=ckpt.get("config", cfg),
            evolution_dim=ckpt.get("evolution_dim"),
            num_edge_heads=ckpt.get("num_edge_heads", 0),
            reference_feature_mode=ckpt.get("reference_feature_mode", "hlca_luca"),
        )
        model.load_state_dict(ckpt["state_dict"])
        model.eval()
        model.to(device)
        return model, ckpt
    except Exception as e:
        print(f"  Warning: could not load checkpoint {checkpoint_path}: {e}")
        return None, None


# --- Run configuration ---
RUN_NAME = "rescue_ablation"
CONTEXT_MODE = "eamist"
USE_GROUPED_LABELS = True

# Paths
DATA_ROOT = Path(os.environ.get("STAGEBRIDGE_DATA_ROOT", "/mnt/e/StageBridge_data"))
OUTPUT_ROOT = Path("outputs/scratch")
REPORT_ROOT = Path("reports")
FIGURE_ROOT = REPORT_ROOT / "figures" / "eamist"
TABLE_ROOT = REPORT_ROOT / "tables" / "eamist"

# Checkpoint search paths (best available EA-MIST models)
EAMIST_CKPT_DIRS = [
    OUTPUT_ROOT / "rescue_ablation_20250608/eamist_benchmark/hlca_luca/eamist",
    OUTPUT_ROOT / "eamist_v1_20260309/eamist_benchmark/hlca_luca/eamist",
    OUTPUT_ROOT / "eamist_3seed_20260310/eamist_benchmark/hlca_luca/eamist",
]

# Compose config
cfg = compose_config(overrides=[
    f"context_model={CONTEXT_MODE}",
    f"run_name={RUN_NAME}",
])

print(f"Run name:         {RUN_NAME}")
print(f"Context mode:     {CONTEXT_MODE}")
print(f"Grouped labels:   {USE_GROUPED_LABELS}")
print(f"Data root:        {DATA_ROOT}")
print(f"Output root:      {OUTPUT_ROOT}")
print(f"Device:           {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"DR backends:      PCA ✓  |  UMAP {'✓' if HAS_UMAP else '✗'}  |  t-SNE ✓  |  PHATE {'✓' if HAS_PHATE else '✗'}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    print(f"VRAM:             {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Part I: Environment Validation

Verify that all required data assets and dependencies are available before proceeding.

In [ ]:
# --- Environment Validation ---
assets = {
    "snRNA merged h5ad": DATA_ROOT / "processed" / "anndata" / "snrna_merged.h5ad",
    "snRNA latent h5ad": DATA_ROOT / "processed" / "anndata" / "snrna_latent_merged.h5ad",
    "Visium merged h5ad": DATA_ROOT / "processed" / "anndata" / "spatial_merged.h5ad",
    "HLCA reference h5ad": DATA_ROOT / "data" / "reference" / "hlca" / "hlca_full_v1.h5ad",
    "WES features": DATA_ROOT / "processed" / "features" / "wes_features.parquet",
    "EA-MIST bags parquet": DATA_ROOT / "processed" / "features" / "eamist_bags.parquet",
}

print(f"PyTorch {torch.__version__}  |  CUDA {'available' if torch.cuda.is_available() else 'NOT available'}")
print()

all_ok = True
for name, path in assets.items():
    exists = path.exists()
    status = "OK" if exists else "MISSING"
    size = f"({path.stat().st_size / 1e6:.0f} MB)" if exists else ""
    if not exists:
        all_ok = False
    print(f"  [{status:>7}] {name}: {path} {size}")

print(f"\nEnvironment gate: {'PASS' if all_ok else 'FAIL — some assets missing'}")

## Part II: Data Preprocessing and Cohort Preview

Load and preview the three data modalities:
- **snRNA-seq**: Single-nucleus RNA from 25 donors across 5 histological stages
- **Visium**: Spatial transcriptomics with tissue coordinates
- **WES**: Whole-exome sequencing features (TMB, driver mutations)

### Embedding analysis
Four dimensionality reduction methods are applied to the snRNA latent space:
- **PCA** — Linear projection with explained variance percentages on each axis
- **UMAP** — Non-linear manifold learning preserving local + global structure
- **t-SNE** — Non-linear embedding emphasizing local cluster separation
- **PHATE** — Potential of Heat-diffusion for Affinity-based Trajectory Embedding (captures continuous transitions)

In [ ]:
# --- Data Preprocessing Overview ---
data_output = run_data_preprocessing_overview(cfg, max_cells_per_stage=256, max_spots_per_stage=256)

# Summary table: modality × obs × features × donors
preprocessing_table = build_dataset_preprocessing_table(data_output)
display(Markdown("### Cohort Summary"))
display(preprocessing_table)

# Stage distribution
snrna_info = data_output.get("snrna", {})
stage_counts = snrna_info.get("stage_counts", {})
if stage_counts:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # snRNA stage counts
    stages = list(stage_counts.keys())
    counts = list(stage_counts.values())
    colors = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(stages)))
    axes[0].barh(stages, counts, color=colors)
    axes[0].set_xlabel("Cell count")
    axes[0].set_title("snRNA-seq cells by stage")
    for i, c in enumerate(counts):
        axes[0].text(c + max(counts) * 0.01, i, f"{c:,}", va="center", fontsize=9)

    # Grouped label distribution (from bags if available)
    from stagebridge.data.luad_evo.stages import GROUPED_STAGE_ORDER, STAGE_TO_GROUP
    grouped = {}
    for stage, count in stage_counts.items():
        g = STAGE_TO_GROUP.get(stage, stage)
        grouped[g] = grouped.get(g, 0) + count
    g_labels = [g for g in GROUPED_STAGE_ORDER if g in grouped]
    g_counts = [grouped[g] for g in g_labels]
    g_colors = ["#4CAF50", "#FF9800", "#F44336"][:len(g_labels)]
    axes[1].barh(g_labels, g_counts, color=g_colors)
    axes[1].set_xlabel("Cell count")
    axes[1].set_title("Grouped ordinal labels")
    for i, c in enumerate(g_counts):
        axes[1].text(c + max(g_counts) * 0.01, i, f"{c:,}", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()

print(f"\nsnRNA: {snrna_info.get('n_cells', 'n/a'):,} cells, {snrna_info.get('n_genes', 'n/a'):,} genes")
print(f"Top HLCA labels: {', '.join(l for l, _ in snrna_info.get('top_labels', [])[:5])}")

In [ ]:
# --- snRNA Embedding: 4-Method Dimensionality Reduction ---
# PCA (with explained variance %), UMAP, t-SNE, PHATE — colored by histological stage.

snrna_latent = snrna_info.get("pca_embedding")  # latent from preprocessing
snrna_stages_arr = snrna_info.get("stages")      # per-cell stage labels

if snrna_latent is not None and snrna_stages_arr is not None:
    snrna_latent = np.asarray(snrna_latent, dtype=np.float32)
    snrna_stages_arr = np.asarray(snrna_stages_arr, dtype=str)

    # ── 4-panel embedding comparison ──
    emb = compute_all_embeddings(snrna_latent, n_subsample=8000)
    idx = emb["_idx"]
    sub_stages = snrna_stages_arr[idx]

    fig = plot_four_embeddings(
        emb, sub_stages, STAGE_COLORS,
        title="snRNA-seq Latent Space — 4 Embedding Methods",
        output_path=FIGURE_ROOT / "fig_snrna_4embeddings.png",
    )
    display(fig); plt.close(fig)

    # ── PCA scree plot (explained variance) ──
    pca_full = PCA(n_components=min(30, snrna_latent.shape[1]), random_state=42)
    pca_full.fit(snrna_latent[idx])
    var_ratio = pca_full.explained_variance_ratio_ * 100
    cum_var = np.cumsum(var_ratio)

    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.bar(range(1, len(var_ratio)+1), var_ratio, color="#1B9E77", edgecolor="white")
    ax1.set_xlabel("Principal Component"); ax1.set_ylabel("Variance Explained (%)")
    ax1.set_title("PCA Scree Plot")
    ax1.axhline(y=5, color="gray", ls="--", alpha=0.5, label="5% threshold")
    ax1.legend(fontsize=8)

    ax2.plot(range(1, len(cum_var)+1), cum_var, "o-", color="#D95F02", lw=2)
    ax2.axhline(y=90, color="gray", ls="--", alpha=0.5, label="90% cumulative")
    n90 = int(np.searchsorted(cum_var, 90.0) + 1)
    ax2.axvline(x=n90, color="#E41A1C", ls=":", alpha=0.7, label=f"{n90} PCs for 90%")
    ax2.set_xlabel("Number of PCs"); ax2.set_ylabel("Cumulative Variance (%)")
    ax2.set_title("Cumulative Variance Explained")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(FIGURE_ROOT / "fig_snrna_pca_scree.png", dpi=300, bbox_inches="tight")
    display(fig2); plt.close(fig2)

    # ── Per-stage density contours on UMAP ──
    umap_coords = emb["UMAP"][0]
    fig3, ax = plt.subplots(figsize=(8, 7))
    for stage in CANONICAL_STAGE_ORDER:
        mask = sub_stages == stage
        if not mask.any():
            continue
        ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1], s=4, alpha=0.3,
                   color=STAGE_COLORS.get(stage, "gray"), label=stage, rasterized=True)
        # KDE contours for each stage
        if mask.sum() > 30:
            try:
                xy = umap_coords[mask].T
                kde = gaussian_kde(xy)
                xmin, xmax = umap_coords[:, 0].min(), umap_coords[:, 0].max()
                ymin, ymax = umap_coords[:, 1].min(), umap_coords[:, 1].max()
                xx, yy = np.mgrid[xmin:xmax:80j, ymin:ymax:80j]
                zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
                ax.contour(xx, yy, zz, levels=3, colors=[STAGE_COLORS.get(stage, "gray")],
                           alpha=0.6, linewidths=1.2)
            except Exception:
                pass
    ax.set_title("UMAP with Stage Density Contours", fontsize=13, fontweight="bold")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    ax.legend(frameon=True, fontsize=9, markerscale=3)
    fig3.tight_layout()
    fig3.savefig(FIGURE_ROOT / "fig_snrna_umap_density.png", dpi=300, bbox_inches="tight")
    display(fig3); plt.close(fig3)

    print(f"✓ {len(idx):,} cells embedded  |  PCA: {n90} PCs for 90% variance")
    print(f"  Variance explained by PC1-PC3: {var_ratio[0]:.1f}%, {var_ratio[1]:.1f}%, {var_ratio[2]:.1f}%")
else:
    print("No precomputed embeddings available; run full preprocessing to generate.")

## Part III: Reference Latent Mapping (HLCA + LuCA)

Two atlas references anchor the niche feature space:
- **HLCA** (Human Lung Cell Atlas) — 13D cosine similarities to healthy lung cell types
- **LuCA** (Lung Cancer Atlas) — 15D cosine similarities to cancer-associated cell types

The alignment gate checks stage probe accuracy, donor leakage, and label coverage.
A good alignment means the latent space preserves biological signal without batch confounding.

In [ ]:
# --- Run Reference Backend (HLCA) ---
reference_output = run_reference(cfg)

# Summary table: backend, latent shape, stage probe accuracy, donor leakage, gate status
ref_summary = build_reference_summary_table(reference_output)
display(Markdown("### Reference Alignment Summary"))
display(ref_summary)

# Extended evaluation: balanced accuracy, centroid distances, neighbor agreement
ref_eval = build_reference_evaluation_table(reference_output)
display(Markdown("### Reference Evaluation Metrics"))
display(ref_eval)

# Top transferred labels
ref_labels = build_reference_label_table(reference_output)
display(Markdown("### Top Transferred HLCA Labels"))
display(ref_labels)

# Alignment gate
diag = reference_output.get("reference", {}).get("diagnostics", {})
gate = diag.get("alignment_gate", {})
print(f"\nAlignment gate: {gate.get('status', 'n/a')} — {gate.get('recommended_action', '')}")

In [ ]:
# --- Reference Alignment Visualization ---
from stagebridge.viz.research_frontend import plot_reference_frontend

fig_ref = plot_reference_frontend(
    reference_output,
    output_path=FIGURE_ROOT / "reference_alignment.png",
)
display(fig_ref); plt.close(fig_ref)
print("Panels: Stage preservation UMAP | Donor leakage probe | Label coverage")

## Part IV: Spatial Deconvolution (Tangram / TACCO / DestVI)

Three spatial mapping methods deconvolve Visium spots into cell-type compositions:

| Method | Approach | Key strength |
|--------|---------|-------------|
| **Tangram** | Optimal transport alignment | Fast, robust baseline |
| **TACCO** | Transfer learning + annotation | Compositional accuracy |
| **DestVI** | Variational inference | Uncertainty quantification |

The provider ladder runs all three, computes QC heuristics, and pairwise agreement.

In [ ]:
# --- Run Spatial Provider Ladder ---
provider_outputs = run_spatial_provider_ladder(
    cfg,
    methods=["tangram", "tacco", "destvi"],
    reference_output=reference_output,
)

# QC heuristic scoring: row sum, max assignment, entropy, diversity
provider_qc = build_spatial_provider_metric_table(provider_outputs)
display(Markdown("### Spatial Provider QC Metrics"))
display(provider_qc)

# Pairwise agreement between providers
provider_agreement = build_spatial_provider_agreement_table(provider_outputs)
display(Markdown("### Provider Pairwise Agreement"))
display(provider_agreement)

# Spatial cell-type maps for the top provider
from stagebridge.viz.spatial import plot_tangram_winner_map, plot_tangram_celltype_maps

top_provider = provider_qc.iloc[0]["method"] if len(provider_qc) > 0 else "tangram"
top_result = provider_outputs.get(top_provider, {})
mapping = top_result.get("mapping_result")

if mapping is not None and mapping.compositions is not None:
    plot_tangram_winner_map(
        mapping.compositions, mapping.feature_names, mapping.coords,
        output_path=FIGURE_ROOT / f"spatial_winner_map_{top_provider}.png",
    )
    plot_tangram_celltype_maps(
        mapping.compositions, mapping.feature_names, mapping.coords,
        output_path=FIGURE_ROOT / f"spatial_celltype_maps_{top_provider}.png",
    )
    print(f"\nSelected provider: {top_provider}")
    print(f"Spots: {mapping.compositions.shape[0]:,} | Cell types: {mapping.compositions.shape[1]}")
else:
    print(f"Spatial mapping not available; check provider ladder output.")

## Part V: EA-MIST Lesion Bags — Construction and Exploration

Each lesion is encoded as a **bag of neighborhoods**. The parquet dataset contains ~639K neighborhoods across 56 lesions from 25 donors.

### Bag features per neighborhood:
- `receiver_embedding` — Central cell latent vector
- `ring_compositions` — Cell-type compositions at 4 spatial radii
- `hlca_features` (13D) — Cosine similarities to HLCA healthy cell types
- `luca_features` (15D) — Cosine similarities to LuCA cancer cell types
- `lr_pathway_summary` — Ligand-receptor pathway activity
- `neighborhood_stats` — Density, diversity, uncertainty

### Grouped ordinal labels:
| Group | Stages | Count | Displacement |
|-------|--------|-------|-------------|
| `early_like` | Normal + AAH | 12 | 0.0 |
| `intermediate_like` | AIS + MIA | 18 | 0.5 |
| `invasive_like` | LUAD | 26 | 1.0 |

In [ ]:
# --- Load and Explore EA-MIST Bags ---
bags_path = DATA_ROOT / "processed" / "features" / "eamist_bags.parquet"

if bags_path.exists():
    bags_df = pd.read_parquet(bags_path)
    print(f"Bags parquet: {bags_df.shape[0]:,} neighborhoods, {bags_df.shape[1]} columns")
    print(f"Lesions:      {bags_df['lesion_id'].nunique()}")
    print(f"Donors:       {bags_df['donor_id'].nunique()}")
    print()

    # Stage distribution
    from stagebridge.data.luad_evo.stages import (
        CANONICAL_STAGE_ORDER, GROUPED_STAGE_ORDER, STAGE_TO_GROUP
    )

    lesion_stages = bags_df.groupby("lesion_id")["stage"].first()
    canonical_counts = lesion_stages.value_counts().reindex(CANONICAL_STAGE_ORDER, fill_value=0)
    grouped_counts = lesion_stages.map(STAGE_TO_GROUP).value_counts().reindex(GROUPED_STAGE_ORDER, fill_value=0)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Canonical stage distribution
    canonical_counts.plot.bar(ax=axes[0], color=plt.cm.YlOrRd(np.linspace(0.2, 0.9, 5)))
    axes[0].set_title("Lesions by canonical stage")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis='x', rotation=45)

    # Grouped distribution
    grouped_counts.plot.bar(ax=axes[1], color=["#4CAF50", "#FF9800", "#F44336"])
    axes[1].set_title("Lesions by grouped label")
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis='x', rotation=45)

    # Neighborhoods per lesion
    nhoods_per_lesion = bags_df.groupby("lesion_id").size()
    nhoods_per_lesion.hist(ax=axes[2], bins=20, color="#2196F3", edgecolor="white")
    axes[2].set_title(f"Neighborhoods per lesion (median={nhoods_per_lesion.median():.0f})")
    axes[2].set_xlabel("Neighborhoods")
    axes[2].set_ylabel("Lesions")

    plt.tight_layout()
    plt.show()

    # Feature dimensions
    feature_cols = {
        "hlca_features": [c for c in bags_df.columns if c.startswith("hlca_")],
        "luca_features": [c for c in bags_df.columns if c.startswith("luca_")],
    }
    for name, cols in feature_cols.items():
        if cols:
            print(f"  {name}: {len(cols)}D")

    display(Markdown("### Lesion-level summary"))
    lesion_summary = bags_df.groupby(["lesion_id", "donor_id", "stage"]).size().reset_index(name="n_neighborhoods")
    lesion_summary["grouped_label"] = lesion_summary["stage"].map(STAGE_TO_GROUP)
    display(lesion_summary.sort_values("stage").head(15))
else:
    print(f"Bags parquet not found at {bags_path}")

### Niche-Level Embedding Analysis

Dimensionality reduction on the **combined atlas feature space** (HLCA 13D + LuCA 15D = 28D) for individual neighborhoods.
Each point represents one spatial neighborhood; coloring by grouped label reveals whether the atlas features encode
stage-discriminative structure at the niche level — before any model aggregation.

| Method | Strengths | Parameters |
|--------|----------|-----------|
| **PCA** | Linear, interpretable, shows variance structure | Explained variance % on axes |
| **UMAP** | Preserves local + global topology | n_neighbors=30, min_dist=0.3 |
| **t-SNE** | Sharp local clusters | perplexity adaptive |
| **PHATE** | Captures continuous transitions / trajectories | PHATE operator, fallback to UMAP |

In [ ]:
# --- Niche-Level 4-Method Embedding (Atlas Features) ---
if bags_path.exists():
    hlca_cols = sorted([c for c in bags_df.columns if c.startswith("hlca_")])
    luca_cols = sorted([c for c in bags_df.columns if c.startswith("luca_")])
    atlas_cols = hlca_cols + luca_cols

    if atlas_cols:
        X_atlas = bags_df[atlas_cols].values.astype(np.float32)
        niche_labels = bags_df["stage"].map(STAGE_TO_GROUP).values

        # Subsample for tractable DR
        niche_emb = compute_all_embeddings(X_atlas, n_subsample=10000, seed=42)
        idx_n = niche_emb["_idx"]
        sub_labels = niche_labels[idx_n]

        # ── 4-panel view by grouped label ──
        fig = plot_four_embeddings(
            niche_emb, sub_labels, GROUP_COLORS,
            title="Niche-Level Atlas Features (28D) — Grouped Labels",
            output_path=FIGURE_ROOT / "fig_niche_4embeddings_grouped.png",
            point_size=5,
        )
        display(fig); plt.close(fig)

        # ── Same embeddings colored by canonical stage ──
        sub_stages_canon = bags_df["stage"].values[idx_n]
        fig2 = plot_four_embeddings(
            niche_emb, sub_stages_canon, STAGE_COLORS,
            title="Niche-Level Atlas Features (28D) — Canonical Stages",
            output_path=FIGURE_ROOT / "fig_niche_4embeddings_canonical.png",
            point_size=5,
        )
        display(fig2); plt.close(fig2)

        # ── UMAP with grouped-label confidence ellipses ──
        umap_niche = niche_emb["UMAP"][0]
        fig3, ax = plt.subplots(figsize=(9, 8))
        for grp in GROUPED_STAGE_ORDER:
            mask = sub_labels == grp
            if not mask.any():
                continue
            ax.scatter(umap_niche[mask, 0], umap_niche[mask, 1], s=4, alpha=0.3,
                       color=GROUP_COLORS[grp], label=grp, rasterized=True)
            confidence_ellipse(umap_niche[mask, 0], umap_niche[mask, 1], ax,
                               n_std=2.0, facecolor=GROUP_COLORS[grp], alpha=0.12,
                               edgecolor=GROUP_COLORS[grp], linewidth=2)
        ax.set_title("UMAP — Niche Atlas Features with 95% Confidence Ellipses",
                      fontsize=12, fontweight="bold")
        ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
        ax.legend(frameon=True, fontsize=10, markerscale=3)
        fig3.tight_layout()
        fig3.savefig(FIGURE_ROOT / "fig_niche_umap_ellipses.png", dpi=300, bbox_inches="tight")
        display(fig3); plt.close(fig3)

        # ── PCA variance breakdown ──
        pca_niche = PCA(n_components=min(20, len(atlas_cols)), random_state=42)
        pca_niche.fit(X_atlas[idx_n])
        var_n = pca_niche.explained_variance_ratio_ * 100
        cum_n = np.cumsum(var_n)

        fig4, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar(range(1, len(var_n)+1), var_n, color="#0E7490", edgecolor="white", label="Individual")
        ax2 = ax.twinx()
        ax2.plot(range(1, len(cum_n)+1), cum_n, "o-", color="#D95F02", lw=2, label="Cumulative")
        ax2.axhline(y=90, color="gray", ls="--", alpha=0.5)
        ax.set_xlabel("Principal Component"); ax.set_ylabel("Variance Explained (%)")
        ax2.set_ylabel("Cumulative %")
        ax.set_title(f"Atlas Feature PCA — {len(atlas_cols)}D input", fontweight="bold")
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
        fig4.tight_layout()
        fig4.savefig(FIGURE_ROOT / "fig_niche_pca_variance.png", dpi=300, bbox_inches="tight")
        display(fig4); plt.close(fig4)

        print(f"✓ {len(idx_n):,} neighborhoods embedded from {len(atlas_cols)}D atlas feature space")
        print(f"  PCA: PC1={var_n[0]:.1f}%, PC1-5 cumulative={cum_n[min(4,len(cum_n)-1)]:.1f}%")
    else:
        print("No atlas feature columns found in bags_df.")
else:
    print("Bags parquet not found.")

### Lesion-Level Embedding Analysis

Aggregated atlas features (mean + std per lesion) projected into 2D. With only 56 lesions, every point
is visible and confidence ellipses show the geometric separation between grouped labels.
Good separation here indicates the atlas features carry lesion-level stage signal even before a classifier is trained.

In [ ]:
# --- Lesion-Level 4-Method Embedding + Confidence Ellipses ---
if bags_path.exists() and atlas_cols:
    # Aggregate: mean + std per lesion
    lesion_mean = bags_df.groupby("lesion_id")[atlas_cols].mean()
    lesion_std  = bags_df.groupby("lesion_id")[atlas_cols].std().fillna(0)
    lesion_meta = bags_df.groupby("lesion_id").agg(
        stage=("stage", "first"), donor_id=("donor_id", "first")
    )
    # Combine mean + std into a single feature matrix (56 × 56D)
    X_lesion = np.hstack([lesion_mean.values, lesion_std.values]).astype(np.float32)
    lesion_groups = lesion_meta["stage"].map(STAGE_TO_GROUP).values
    lesion_stages = lesion_meta["stage"].values
    lesion_ids = lesion_mean.index.values

    # Compute all embeddings (no subsampling needed — only 56 lesions)
    lesion_emb = compute_all_embeddings(X_lesion, n_subsample=999, seed=42)

    # ── 4-panel by grouped label with ellipses ──
    methods = ["PCA", "UMAP", "t-SNE", "PHATE"]
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    for ax, method in zip(axes, methods):
        coords, meta = lesion_emb[method]
        subtitle = f"{method}  {meta}" if meta else method
        for grp in GROUPED_STAGE_ORDER:
            mask = lesion_groups == grp
            if not mask.any():
                continue
            ax.scatter(coords[mask, 0], coords[mask, 1], s=60, alpha=0.75,
                       color=GROUP_COLORS[grp], label=grp, edgecolors="white", linewidths=0.8,
                       zorder=3)
            confidence_ellipse(coords[mask, 0], coords[mask, 1], ax, n_std=2.0,
                               facecolor=GROUP_COLORS[grp], alpha=0.10,
                               edgecolor=GROUP_COLORS[grp], linewidth=2, zorder=2)
        ax.set_title(subtitle, fontsize=11, fontweight="bold")
        ax.set_xlabel(f"{method} 1"); ax.set_ylabel(f"{method} 2")
        ax.legend(frameon=True, fontsize=8, markerscale=1.2)
    fig.suptitle("Lesion-Level Atlas Features (mean+std, 56D) — Grouped Labels",
                 fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(FIGURE_ROOT / "fig_lesion_4embeddings_grouped.png", dpi=300, bbox_inches="tight")
    display(fig); plt.close(fig)

    # ── Annotated UMAP with lesion IDs ──
    umap_lesion = lesion_emb["UMAP"][0]
    fig2, ax = plt.subplots(figsize=(10, 9))
    for grp in GROUPED_STAGE_ORDER:
        mask = lesion_groups == grp
        ax.scatter(umap_lesion[mask, 0], umap_lesion[mask, 1], s=80, alpha=0.8,
                   color=GROUP_COLORS[grp], label=grp, edgecolors="white", linewidths=1, zorder=3)
        confidence_ellipse(umap_lesion[mask, 0], umap_lesion[mask, 1], ax, n_std=2.0,
                           facecolor=GROUP_COLORS[grp], alpha=0.08,
                           edgecolor=GROUP_COLORS[grp], linewidth=2.5, zorder=2)
    # Annotate each point
    for i, lid in enumerate(lesion_ids):
        ax.annotate(str(lid)[:8], (umap_lesion[i, 0], umap_lesion[i, 1]),
                    fontsize=5.5, alpha=0.7, ha="center", va="bottom",
                    xytext=(0, 4), textcoords="offset points")
    ax.set_title("Lesion-Level UMAP with IDs and 95% Confidence Ellipses",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    ax.legend(frameon=True, fontsize=10, markerscale=1.5)
    fig2.tight_layout()
    fig2.savefig(FIGURE_ROOT / "fig_lesion_umap_annotated.png", dpi=300, bbox_inches="tight")
    display(fig2); plt.close(fig2)

    # ── 3D PCA scatter ──
    pca_3d = lesion_emb.get("pca_3d")
    if pca_3d is not None and pca_3d.shape[1] >= 3:
        fig3 = plot_3d_embedding(
            pca_3d, labels=lesion_groups,
            title="Lesion-Level PCA (3D) — Grouped Labels",
            output_path=FIGURE_ROOT / "fig_lesion_pca3d.png",
            point_size=50, alpha=0.8,
        )
        display(fig3); plt.close(fig3)

    pca_var_lesion = lesion_emb["pca_var"]
    print(f"✓ {len(lesion_ids)} lesions embedded from {X_lesion.shape[1]}D (mean+std atlas features)")
    print(f"  PCA: PC1={pca_var_lesion[0]:.1f}%, PC2={pca_var_lesion[1]:.1f}%, PC3={pca_var_lesion[2]:.1f}%")
else:
    print("Bags parquet or atlas columns not available.")

## Part V-B: EA-MIST Architecture and Token Types

The EA-MIST model processes each local niche through a **7-token transformer** that captures distinct biological signal channels:

| Token Type | Index | Source | Biological Role |
|------------|-------|--------|-----------------|
| **Receiver** | 0 | Epithelial cell expression + state embedding | Central cell identity and transcriptomic state |
| **Ring** (x4) | 1 | Sender composition per distance ring | Spatial neighborhood structure |
| **HLCA atlas** | 2 | Cosine similarity to Human Lung Cell Atlas | Reference positioning (healthy cell types) |
| **LuCA atlas** | 3 | Cosine similarity to Lung Cancer Atlas | Reference positioning (tumor cell types) |
| **LR pathway** | 4 | Ligand-receptor pathway summary | Cell-cell communication signals |
| **Niche stats** | 5 | Neighborhood summary statistics | Microenvironment characterization |
| **Atlas contrast** | 6 | `[h, l, l-h, h*l, |l-h|]` MLP | Cross-atlas divergence signal |

### Architecture flow
```
Local Niches (N per lesion)
  -> LocalNicheTokenizer (7 tokens each)
  -> 2-layer Local Transformer -> neighborhood embeddings (N x D)
  -> Prototype Bottleneck (K=16 learned motifs) -> aligned embeddings
  -> 2-layer Set Transformer (ISAB + PMA) -> lesion embedding (1 x D)
  -> Evolution Branch (gated fusion) -> fused embedding
  -> Distribution-Aware Pooling (niche transition scores -> 7 summary stats)
  -> Multitask Heads: {stage classification, displacement regression, edge prediction}
```

In [ ]:
# --- Fig 24: EA-MIST Architecture Diagram with Token Types ---
fig_arch, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(-1, 17)
ax.set_ylim(-1, 11)
ax.set_aspect("equal")
ax.axis("off")

# Token type boxes (left column)
for i, (name, color) in enumerate(zip(TOKEN_TYPE_NAMES, TOKEN_TYPE_COLORS)):
    y = 9.5 - i * 1.3
    box = FancyBboxPatch((0.2, y - 0.35), 3.0, 0.7, boxstyle="round,pad=0.1",
                         facecolor=color, alpha=0.25, edgecolor=color, linewidth=2)
    ax.add_patch(box)
    ax.text(1.7, y, f"[{i}] {name}", ha="center", va="center", fontsize=10,
            fontweight="bold", color=color)

# Arrow from tokens → Local Transformer
ax.annotate("", xy=(4.5, 5.0), xytext=(3.4, 5.0),
            arrowprops=dict(arrowstyle="->", lw=2, color="#333"))
ax.text(3.9, 5.4, "tokenize", ha="center", va="center", fontsize=8, style="italic")

# Module boxes (flow right)
modules = [
    (5.0, 4.0, 2.8, 2.0, "Local\nTransformer\n(2-layer SAB)", "#1f77b4"),
    (8.5, 4.0, 2.5, 2.0, "Prototype\nBottleneck\n(K=16)", "#ff7f0e"),
    (11.5, 4.0, 2.5, 2.0, "Set\nTransformer\n(ISAB+PMA)", "#2ca02c"),
    (5.0, 0.5, 2.8, 1.5, "Evolution\nBranch\n(gated)", "#9467bd"),
    (8.5, 0.5, 2.5, 1.5, "Dist.-Aware\nPooling\n(7 stats)", "#d62728"),
    (11.5, 0.5, 2.5, 1.5, "Multitask\nHeads\n(stage/disp/edge)", "#8c564b"),
]
for x, y, w, h, label, color in modules:
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15",
                         facecolor=color, alpha=0.15, edgecolor=color, linewidth=2.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=10,
            fontweight="bold", color=color)

# Arrows between modules
arrow_kw = dict(arrowstyle="-|>", lw=2, color="#555")
ax.annotate("", xy=(8.3, 5.0), xytext=(7.8, 5.0), arrowprops=arrow_kw)
ax.annotate("", xy=(11.3, 5.0), xytext=(11.0, 5.0), arrowprops=arrow_kw)
# Down from Set Transformer to heads row
ax.annotate("", xy=(12.75, 2.2), xytext=(12.75, 3.8), arrowprops=arrow_kw)
# Evolution branch input (from left)
ax.annotate("", xy=(4.8, 1.25), xytext=(4.0, 1.25),
            arrowprops=dict(arrowstyle="-|>", lw=1.5, color="#9467bd", ls="--"))
ax.text(3.5, 1.7, "WES/CNA\nfeatures", ha="center", va="center", fontsize=8, color="#9467bd")
# Evolution → Dist pooling
ax.annotate("", xy=(8.3, 1.25), xytext=(7.8, 1.25), arrowprops=arrow_kw)
# Dist pooling → Heads
ax.annotate("", xy=(11.3, 1.25), xytext=(11.0, 1.25), arrowprops=arrow_kw)

# Output labels
out_labels = [("Stage\nlogits", 14.5, 1.7), ("Displacement", 14.5, 1.0), ("Edge\nlogits", 14.5, 0.3)]
for label, x, y in out_labels:
    ax.text(x, y, label, ha="center", va="center", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="#eee", edgecolor="#888"))
ax.annotate("", xy=(14.0, 1.25), xytext=(14.0, 1.25), arrowprops=arrow_kw)

# Title and annotations
ax.set_title("EA-MIST v1.5 Architecture: 7-Token Local Niche Transformer + Set Transformer",
             fontsize=14, fontweight="bold", pad=15)
ax.text(6.4, 9.8, "N local niches per lesion", fontsize=11, ha="center",
        style="italic", color="#555",
        bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))

fig_arch.tight_layout()
fig_arch.savefig(FIGURE_ROOT / "fig24_architecture_diagram.png", dpi=300, bbox_inches="tight")
fig_arch.savefig(FIGURE_ROOT / "fig24_architecture_diagram.pdf", bbox_inches="tight")
display(fig_arch); plt.close(fig_arch)
print("✓ fig24_architecture_diagram.png/pdf")

## Part V-C: Model Interpretability — Checkpoint Loading

Load the best available trained EA-MIST checkpoint and run a forward pass with `return_attention=True` to extract:
- **Prototype assignment weights** (B, N, K=16) — which niche motif each neighborhood belongs to
- **Local attention weights** — which token types the local transformer attends to
- **Lesion attention weights** — which neighborhoods matter most for the lesion-level prediction
- **Niche transition scores** (B, N) — per-niche transition activity

In [ ]:
# --- Load EA-MIST checkpoint and data for interpretability ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# Find best available EA-MIST checkpoint
eamist_model = None
eamist_ckpt = None
ckpt_path_used = None

for ckpt_dir in EAMIST_CKPT_DIRS:
    if not ckpt_dir.exists():
        continue
    candidates = sorted(ckpt_dir.glob("fold_*/seed_*/best_checkpoint.pt"))
    if not candidates:
        continue
    ckpt_path_used = candidates[0]
    eamist_model, eamist_ckpt = load_eamist_checkpoint(ckpt_path_used, cfg, device)
    if eamist_model is not None:
        break

if eamist_model is None:
    print("WARNING: No trained EA-MIST checkpoint found. Model interpretability figures will use")
    print("         architecture-only visualizations and synthetic demonstrations.")
    HAS_EAMIST_CKPT = False
else:
    HAS_EAMIST_CKPT = True
    print(f"✓ Loaded EA-MIST from {ckpt_path_used}")
    n_params = sum(p.numel() for p in eamist_model.parameters())
    print(f"  Model family:     {eamist_ckpt['model_family']}")
    print(f"  Parameters:       {n_params:,}")
    print(f"  Hidden dim:       {eamist_model.hidden_dim}")
    print(f"  Prototypes:       {'yes (K=16)' if eamist_model.prototype_bottleneck is not None else 'no'}")
    print(f"  Evolution branch: {'yes' if eamist_model.evolution_branch is not None else 'no'}")
    print(f"  Dist. pooling:    {'yes' if eamist_model.niche_transition_head is not None else 'no'}")
    print(f"  Val metrics:      {eamist_ckpt.get('val_metrics', {})}")

# Load bags from the canonical prebuilt parquet
from stagebridge.data.luad_evo.neighborhood_builder import build_lesion_bags_from_parquet
interp_batch = None
interp_bags_list = None
interp_stages = None

eamist_bag_parquet = DATA_ROOT / "processed" / "features" / "eamist_bags.parquet"
if HAS_EAMIST_CKPT and eamist_bag_parquet.exists():
    try:
        build_result = build_lesion_bags_from_parquet(eamist_bag_parquet)
        all_bags = build_result.bags
        # Sample up to 12 diverse lesions (4 per group)
        bag_stage_map = {b.lesion_id: b.stage for b in all_bags}
        selected = []
        for grp in GROUPED_STAGE_ORDER:
            grp_bags = [b for b in all_bags if STAGE_TO_GROUP.get(b.stage) == grp]
            selected.extend(grp_bags[:4])
        if not selected:
            selected = all_bags[:8]
        interp_bags_list = selected
        interp_stages = [b.stage for b in selected]

        # Subsample neighborhoods for memory efficiency
        ds = LesionBagDataset(selected, max_neighborhoods=256)
        batch_bags = [ds[i] for i in range(len(ds))]
        interp_batch = collate_lesion_bags(batch_bags)
        interp_batch = interp_batch.to(device)
        print(f"\n  ✓ Batch: {len(selected)} lesions, max {interp_batch.receiver_embeddings.shape[1]} neighborhoods")
        print(f"    Stages: {interp_stages}")
    except Exception as e:
        print(f"  Warning: Could not build batch: {e}")
        import traceback; traceback.print_exc()

# Run forward pass with attention
eamist_output = None
if HAS_EAMIST_CKPT and interp_batch is not None:
    with torch.no_grad():
        try:
            eamist_output = eamist_model(interp_batch, return_attention=True)
            print(f"\n  ✓ Forward pass complete:")
            print(f"    local_embeddings:  {eamist_output.local_embeddings.shape}")
            print(f"    lesion_embedding:  {eamist_output.lesion_embedding.shape}")
            print(f"    stage_logits:      {eamist_output.stage_logits.shape}")
            if eamist_output.prototype_output is not None:
                po = eamist_output.prototype_output
                print(f"    prototype_assign:  {po.assignment_weights.shape}")
                print(f"    prototype_comp:    {po.prototype_composition.shape}")
                print(f"    prototype_bank:    {po.prototype_bank.shape}")
            if eamist_output.local_attention is not None:
                if isinstance(eamist_output.local_attention, dict):
                    print(f"    local_attention:   dict with keys {list(eamist_output.local_attention.keys())}")
                else:
                    print(f"    local_attention:   {eamist_output.local_attention.shape}")
            if eamist_output.lesion_attention is not None:
                if isinstance(eamist_output.lesion_attention, dict):
                    print(f"    lesion_attention:  dict with keys {list(eamist_output.lesion_attention.keys())}")
                else:
                    print(f"    lesion_attention:  {eamist_output.lesion_attention.shape}")
            if eamist_output.niche_transition_scores is not None:
                print(f"    niche_scores:      {eamist_output.niche_transition_scores.shape}")
        except Exception as e:
            print(f"  Warning: Forward pass failed: {e}")
            import traceback; traceback.print_exc()
            eamist_output = None

### Prototype Bottleneck Analysis

The prototype bottleneck compresses each neighborhood embedding into a soft assignment over **K=16 learned motif prototypes**. Each prototype captures a recurring niche microenvironment pattern. We visualize:
1. **Prototype composition heatmap** — per-lesion mean assignment weights, clustered by stage
2. **Prototype bank PCA** — learned prototype vectors in 2D
3. **Prototype occupancy** — how uniformly neighborhoods distribute across prototypes

In [ ]:
# --- Fig 25: Prototype Bottleneck Analysis (3-panel) ---
if eamist_output is not None and eamist_output.prototype_output is not None:
    po = eamist_output.prototype_output
    proto_comp = po.prototype_composition.cpu().numpy()  # (B, K)
    proto_bank = po.prototype_bank.detach().cpu().numpy()  # (K, D)
    assign_w = po.assignment_weights.cpu().numpy()  # (B, N, K)
    mask = interp_batch.neighborhood_mask.cpu().numpy()  # (B, N)
    B, K = proto_comp.shape

    fig_proto, axes = plt.subplots(1, 3, figsize=(20, 6),
                                    gridspec_kw={"width_ratios": [2.5, 1, 1]})

    # Panel A: Prototype composition heatmap (lesions × prototypes)
    ax = axes[0]
    # Color the row labels by stage group
    stage_groups = [STAGE_TO_GROUP.get(s, "unknown") for s in interp_stages]
    row_labels = [f"{interp_batch.lesion_ids[i][:12]}  ({interp_stages[i]})"
                  for i in range(B)]
    row_colors = [GROUP_COLORS.get(g, "#999") for g in stage_groups]

    im = ax.imshow(proto_comp, aspect="auto", cmap="YlOrRd", interpolation="nearest")
    ax.set_xticks(range(K))
    ax.set_xticklabels([f"P{k}" for k in range(K)], fontsize=8)
    ax.set_yticks(range(B))
    ax.set_yticklabels(row_labels, fontsize=8)
    for i, color in enumerate(row_colors):
        ax.get_yticklabels()[i].set_color(color)
    plt.colorbar(im, ax=ax, shrink=0.7, label="Mean assignment weight")
    ax.set_xlabel("Prototype index")
    ax.set_ylabel("Lesion (stage)")
    ax.set_title("A. Prototype Composition by Lesion", fontweight="bold")

    # Panel B: Prototype bank PCA (K points in 2D)
    ax = axes[1]
    pca_proto = PCA(n_components=2).fit_transform(proto_bank)
    for k in range(K):
        ax.scatter(pca_proto[k, 0], pca_proto[k, 1], s=120, c=[PROTO_CMAP(k)],
                   edgecolors="black", linewidths=1.2, zorder=3)
        ax.annotate(f"P{k}", (pca_proto[k, 0], pca_proto[k, 1]),
                    fontsize=7, fontweight="bold", ha="center", va="bottom",
                    xytext=(0, 6), textcoords="offset points")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.set_title("B. Prototype Bank (PCA)", fontweight="bold")
    ax.grid(True, alpha=0.3)

    # Panel C: Global prototype occupancy (mean assignment mass per prototype)
    ax = axes[2]
    # Compute occupancy: for each prototype, sum of assignment weights across all valid neighborhoods
    occupancy = np.zeros(K)
    for b in range(B):
        valid = mask[b].astype(bool)
        occupancy += assign_w[b, valid].sum(axis=0)
    occupancy /= occupancy.sum()

    bars = ax.bar(range(K), occupancy, color=[PROTO_CMAP(k) for k in range(K)],
                  edgecolor="black", linewidth=0.8)
    ax.set_xticks(range(K))
    ax.set_xticklabels([f"P{k}" for k in range(K)], fontsize=8)
    ax.set_ylabel("Fractional occupancy")
    ax.set_title("C. Prototype Occupancy", fontweight="bold")
    ax.axhline(1.0/K, color="gray", ls="--", alpha=0.5, label=f"Uniform (1/{K})")
    ax.legend(fontsize=8)

    fig_proto.suptitle("EA-MIST Prototype Bottleneck: Learned Niche Motifs (K=16)",
                       fontsize=14, fontweight="bold")
    fig_proto.tight_layout(rect=[0, 0, 1, 0.94])
    fig_proto.savefig(FIGURE_ROOT / "fig25_prototype_analysis.png", dpi=300, bbox_inches="tight")
    fig_proto.savefig(FIGURE_ROOT / "fig25_prototype_analysis.pdf", bbox_inches="tight")
    display(fig_proto); plt.close(fig_proto)
    print("✓ fig25_prototype_analysis.png/pdf")

    # Prototype diversity metric
    proto_sim = proto_bank @ proto_bank.T
    proto_norms = np.linalg.norm(proto_bank, axis=1, keepdims=True)
    cosine_sim = proto_sim / (proto_norms @ proto_norms.T + 1e-8)
    off_diag = cosine_sim[~np.eye(K, dtype=bool)]
    print(f"\n  Prototype cosine similarity (off-diagonal): mean={off_diag.mean():.3f}, "
          f"max={off_diag.max():.3f}, std={off_diag.std():.3f}")
    entropy = -np.sum(occupancy * np.log(occupancy + 1e-8))
    max_entropy = np.log(K)
    print(f"  Occupancy entropy: {entropy:.3f} / {max_entropy:.3f} (max) = {entropy/max_entropy:.1%} utilization")
else:
    print("Prototype analysis requires a trained EA-MIST checkpoint with prototypes.")
    print("Skipping Fig 25.")

### Attention Weight Analysis

The local niche transformer uses multi-head self-attention over the 7 token types. By extracting attention weights we can measure **which biological channels the model focuses on** when encoding each neighborhood.

We also examine the **lesion-level attention** from the Set Transformer's PMA/ISAB blocks to identify which neighborhoods are most important for the final lesion classification.

In [ ]:
# --- Fig 26: Attention Weight Analysis (2-panel) ---
# Token order in local niche: [receiver, ring0, ring1, ring2, ring3, hlca, luca, lr, stats, (contrast)]
LOCAL_TOKEN_LABELS = ["Receiver", "Ring-0", "Ring-1", "Ring-2", "Ring-3",
                       "HLCA", "LuCA", "LR path", "Stats"]

if eamist_output is not None:
    fig_attn, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Panel A: Local attention — average attention TO each token type
    ax = axes[0]
    local_attn = eamist_output.local_attention
    if local_attn is not None:
        if isinstance(local_attn, dict):
            # ISAB returns dict; use "inducing_to_tokens" or first available
            attn_tensor = list(local_attn.values())[0]
        else:
            attn_tensor = local_attn
        attn_np = attn_tensor.cpu().numpy()
        # Shape: (B*N, H, T, T) or (B*N, T, T)
        if attn_np.ndim == 4:
            # Average over heads
            attn_np = attn_np.mean(axis=1)  # (B*N, T, T)
        # Average attention received by each token (column mean)
        T = min(attn_np.shape[-1], len(LOCAL_TOKEN_LABELS))
        attn_to_tokens = attn_np[:, :T, :T].mean(axis=(0, 1))  # (T,)
        labels_used = LOCAL_TOKEN_LABELS[:T]
        colors_used = TOKEN_TYPE_COLORS[:T]

        bars = ax.barh(range(T), attn_to_tokens, color=colors_used, edgecolor="black", linewidth=0.8)
        ax.set_yticks(range(T))
        ax.set_yticklabels(labels_used, fontsize=10)
        ax.set_xlabel("Mean attention weight (received)", fontsize=11)
        ax.set_title("A. Token-Type Importance\n(local transformer attention)", fontweight="bold")
        ax.invert_yaxis()
        for i, v in enumerate(attn_to_tokens):
            ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)
    else:
        ax.text(0.5, 0.5, "Local attention not available\n(model may not support return_attention)",
                ha="center", va="center", transform=ax.transAxes, fontsize=11)
        ax.set_title("A. Token-Type Importance", fontweight="bold")

    # Panel B: Lesion-level attention — neighborhood importance distribution
    ax = axes[1]
    lesion_attn = eamist_output.lesion_attention
    if lesion_attn is not None:
        if isinstance(lesion_attn, dict):
            attn_l = list(lesion_attn.values())[0]
        else:
            attn_l = lesion_attn
        attn_l_np = attn_l.cpu().numpy()
        mask_np = interp_batch.neighborhood_mask.cpu().numpy()
        B = mask_np.shape[0]

        # For each lesion, get the PMA attention weights over neighborhoods
        # Shape could be (B, H, 1, N) for PMA
        if attn_l_np.ndim == 4:
            attn_l_np = attn_l_np.mean(axis=1).squeeze(1)  # (B, N)
        elif attn_l_np.ndim == 3:
            attn_l_np = attn_l_np.mean(axis=1)  # (B, N)

        # Plot attention distribution per lesion, colored by stage
        for b in range(B):
            valid = mask_np[b].astype(bool)
            weights = attn_l_np[b, valid]
            group = STAGE_TO_GROUP.get(interp_stages[b], "unknown")
            ax.plot(sorted(weights, reverse=True), color=GROUP_COLORS.get(group, "#999"),
                    alpha=0.7, linewidth=1.5, label=interp_stages[b] if b < 5 else None)
        ax.set_xlabel("Neighborhood rank (by attention weight)", fontsize=11)
        ax.set_ylabel("Attention weight", fontsize=11)
        ax.set_title("B. Lesion-Level Neighborhood Importance\n(Set Transformer attention)", fontweight="bold")
        # Deduplicated legend
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), fontsize=9, frameon=True)
    else:
        ax.text(0.5, 0.5, "Lesion attention not available",
                ha="center", va="center", transform=ax.transAxes, fontsize=11)
        ax.set_title("B. Neighborhood Importance", fontweight="bold")

    fig_attn.suptitle("EA-MIST Attention Analysis: What the Transformer Learns to Focus On",
                      fontsize=14, fontweight="bold")
    fig_attn.tight_layout(rect=[0, 0, 1, 0.93])
    fig_attn.savefig(FIGURE_ROOT / "fig26_attention_analysis.png", dpi=300, bbox_inches="tight")
    fig_attn.savefig(FIGURE_ROOT / "fig26_attention_analysis.pdf", bbox_inches="tight")
    display(fig_attn); plt.close(fig_attn)
    print("✓ fig26_attention_analysis.png/pdf")
else:
    print("Attention analysis requires a trained EA-MIST checkpoint. Skipping Fig 26.")

### Niche Transition Scores

When distribution-aware pooling is enabled, EA-MIST computes a **per-niche scalar transition score** that reflects how "transition-active" each microenvironment is. These scores are summarized into 7 distribution statistics (mean, std, min, max, q25, q50, q75) and appended to the lesion embedding.

**Biological expectation**: Lesions at active boundaries (e.g., AIS/MIA) should show higher transition score variance, while normal tissue should be uniformly low.

In [ ]:
# --- Fig 27: Niche Transition Score Analysis (2-panel) ---
if eamist_output is not None and eamist_output.niche_transition_scores is not None:
    nts = eamist_output.niche_transition_scores.cpu().numpy()  # (B, N)
    mask_np = interp_batch.neighborhood_mask.cpu().numpy()
    B = nts.shape[0]

    fig_nts, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # Panel A: Transition score distributions by stage group (violin)
    ax = axes[0]
    score_records = []
    for b in range(B):
        valid = mask_np[b].astype(bool)
        scores = nts[b, valid]
        scores = scores[np.isfinite(scores)]
        grp = STAGE_TO_GROUP.get(interp_stages[b], "unknown")
        for s in scores:
            score_records.append({"group": grp, "stage": interp_stages[b], "score": float(s)})
    score_df = pd.DataFrame(score_records)

    if len(score_df) > 0:
        parts = ax.violinplot(
            [score_df[score_df["group"] == g]["score"].values for g in GROUPED_STAGE_ORDER
             if g in score_df["group"].values],
            showmedians=True, showextrema=True
        )
        present_groups = [g for g in GROUPED_STAGE_ORDER if g in score_df["group"].values]
        for i, (pc, grp) in enumerate(zip(parts["bodies"], present_groups)):
            pc.set_facecolor(GROUP_COLORS[grp])
            pc.set_alpha(0.6)
        ax.set_xticks(range(1, len(present_groups) + 1))
        ax.set_xticklabels([g.replace("_like", "") for g in present_groups], fontsize=10)
        ax.set_ylabel("Transition score", fontsize=11)
        ax.set_title("A. Niche Transition Scores by Stage Group", fontweight="bold")

    # Panel B: Per-lesion score statistics (mean ± std)
    ax = axes[1]
    lesion_stats = []
    for b in range(B):
        valid = mask_np[b].astype(bool)
        scores = nts[b, valid]
        scores = scores[np.isfinite(scores)]
        grp = STAGE_TO_GROUP.get(interp_stages[b], "unknown")
        lesion_stats.append({
            "lesion": interp_batch.lesion_ids[b][:12],
            "stage": interp_stages[b],
            "group": grp,
            "mean": scores.mean() if len(scores) > 0 else 0,
            "std": scores.std() if len(scores) > 1 else 0,
        })
    ls_df = pd.DataFrame(lesion_stats)
    if len(ls_df) > 0:
        colors = [GROUP_COLORS.get(g, "#999") for g in ls_df["group"]]
        ax.barh(range(len(ls_df)), ls_df["mean"], xerr=ls_df["std"],
                color=colors, edgecolor="black", linewidth=0.8, capsize=3)
        ax.set_yticks(range(len(ls_df)))
        ax.set_yticklabels([f"{r['lesion']} ({r['stage']})" for _, r in ls_df.iterrows()], fontsize=8)
        ax.set_xlabel("Mean niche transition score", fontsize=11)
        ax.set_title("B. Per-Lesion Transition Activity", fontweight="bold")
        ax.invert_yaxis()

    fig_nts.suptitle("Distribution-Aware Pooling: Per-Niche Transition Scores",
                     fontsize=14, fontweight="bold")
    fig_nts.tight_layout(rect=[0, 0, 1, 0.93])
    fig_nts.savefig(FIGURE_ROOT / "fig27_niche_transition_scores.png", dpi=300, bbox_inches="tight")
    fig_nts.savefig(FIGURE_ROOT / "fig27_niche_transition_scores.pdf", bbox_inches="tight")
    display(fig_nts); plt.close(fig_nts)
    print("✓ fig27_niche_transition_scores.png/pdf")
elif eamist_output is not None:
    print("Niche transition scores not available (model may not use distribution-aware pooling).")
    print("Skipping Fig 27.")
else:
    print("Niche transition analysis requires a trained EA-MIST checkpoint. Skipping Fig 27.")

### Learned Representation Analysis

The model's learned representations should capture biologically meaningful structure. We examine:
1. **Lesion embedding space** — PCA/UMAP of the model's internal lesion representations, colored by stage
2. **Stage prediction confidence** — how confident the model is in its predictions, and whether ordinal neighbors (e.g., AIS vs MIA) are closer than distant stages
3. **Prototype-stage association** — which prototypes preferentially appear in each stage group

In [ ]:
# --- Fig 28: Learned Representation Analysis (3-panel) ---
if eamist_output is not None:
    lesion_embs = eamist_output.lesion_embedding.cpu().numpy()  # (B, D)
    stage_logits = eamist_output.stage_logits.cpu().numpy()  # (B, C)
    B, D = lesion_embs.shape
    C = stage_logits.shape[1]

    fig_repr, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Panel A: Lesion embedding PCA colored by stage
    ax = axes[0]
    if B >= 3:
        pca_le = PCA(n_components=2).fit_transform(lesion_embs)
        for stage in CANONICAL_STAGE_ORDER:
            mask = np.array(interp_stages) == stage
            if not mask.any():
                continue
            ax.scatter(pca_le[mask, 0], pca_le[mask, 1], s=120, alpha=0.8,
                       color=STAGE_COLORS.get(stage, "#999"), label=stage,
                       edgecolors="white", linewidths=1.5, zorder=3)
        # Add lesion ID labels
        for i in range(B):
            ax.annotate(interp_batch.lesion_ids[i][:8], (pca_le[i, 0], pca_le[i, 1]),
                        fontsize=6, alpha=0.7, ha="center", va="bottom",
                        xytext=(0, 5), textcoords="offset points")
        ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
        ax.legend(fontsize=8, frameon=True)
    ax.set_title("A. Learned Lesion Embeddings (PCA)", fontweight="bold")

    # Panel B: Stage prediction probabilities (heatmap)
    ax = axes[1]
    probs = np.exp(stage_logits) / np.exp(stage_logits).sum(axis=1, keepdims=True)  # softmax
    stage_labels = CANONICAL_STAGE_ORDER[:C]
    row_labels = [f"{interp_batch.lesion_ids[i][:10]} ({interp_stages[i]})" for i in range(B)]
    im = ax.imshow(probs, aspect="auto", cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks(range(C))
    ax.set_xticklabels(stage_labels, fontsize=9)
    ax.set_yticks(range(B))
    ax.set_yticklabels(row_labels, fontsize=8)
    # Annotate cells
    for i in range(B):
        for j in range(C):
            color = "white" if probs[i, j] > 0.5 else "black"
            ax.text(j, i, f"{probs[i,j]:.2f}", ha="center", va="center",
                    fontsize=8, color=color, fontweight="bold" if probs[i,j] > 0.3 else "normal")
    # Highlight true class
    for i in range(B):
        true_idx = CANONICAL_STAGE_ORDER.index(interp_stages[i]) if interp_stages[i] in CANONICAL_STAGE_ORDER else -1
        if 0 <= true_idx < C:
            rect = plt.Rectangle((true_idx - 0.5, i - 0.5), 1, 1,
                                  fill=False, edgecolor="red", linewidth=2.5)
            ax.add_patch(rect)
    plt.colorbar(im, ax=ax, shrink=0.7, label="P(stage)")
    ax.set_xlabel("Predicted stage")
    ax.set_title("B. Stage Prediction Probabilities\n(red = true class)", fontweight="bold")

    # Panel C: Prototype-stage association heatmap
    ax = axes[2]
    if eamist_output.prototype_output is not None:
        proto_comp = eamist_output.prototype_output.prototype_composition.cpu().numpy()  # (B, K)
        K = proto_comp.shape[1]
        # Group by stage
        stage_proto = {}
        for grp in GROUPED_STAGE_ORDER:
            grp_mask = np.array([STAGE_TO_GROUP.get(s) == grp for s in interp_stages])
            if grp_mask.any():
                stage_proto[grp] = proto_comp[grp_mask].mean(axis=0)
        if stage_proto:
            assoc_matrix = np.array([stage_proto[g] for g in stage_proto])
            im2 = ax.imshow(assoc_matrix, aspect="auto", cmap="YlOrRd", interpolation="nearest")
            ax.set_xticks(range(K))
            ax.set_xticklabels([f"P{k}" for k in range(K)], fontsize=7)
            ax.set_yticks(range(len(stage_proto)))
            ax.set_yticklabels([g.replace("_like", "") for g in stage_proto], fontsize=10)
            plt.colorbar(im2, ax=ax, shrink=0.7, label="Mean composition")
            ax.set_xlabel("Prototype index")
            ax.set_title("C. Prototype-Stage Association", fontweight="bold")
    else:
        ax.text(0.5, 0.5, "No prototype data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("C. Prototype-Stage Association", fontweight="bold")

    fig_repr.suptitle("EA-MIST Learned Representations and Predictions",
                      fontsize=14, fontweight="bold")
    fig_repr.tight_layout(rect=[0, 0, 1, 0.93])
    fig_repr.savefig(FIGURE_ROOT / "fig28_learned_representations.png", dpi=300, bbox_inches="tight")
    fig_repr.savefig(FIGURE_ROOT / "fig28_learned_representations.pdf", bbox_inches="tight")
    display(fig_repr); plt.close(fig_repr)
    print("✓ fig28_learned_representations.png/pdf")

    # Print prediction summary
    pred_classes = probs.argmax(axis=1)
    true_classes = [CANONICAL_STAGE_ORDER.index(s) if s in CANONICAL_STAGE_ORDER else -1 for s in interp_stages]
    correct = sum(1 for p, t in zip(pred_classes, true_classes) if p == t)
    print(f"\n  Prediction accuracy on interpretability batch: {correct}/{B} ({correct/B:.0%})")
    # Ordinal displacement
    displ = eamist_output.displacement.cpu().numpy().ravel()
    print(f"  Displacement predictions: {', '.join(f'{d:.2f}' for d in displ)}")
else:
    print("Learned representation analysis requires a trained EA-MIST checkpoint. Skipping Fig 28.")

## Part V-D: Biological Grounding — Communication Priors and LR Network

EA-MIST incorporates **24 curated ligand-receptor (L-R) priors** from LUAD biology, organized into 9 signaling families. These priors inform the LR pathway token and connect to **6 receiver programs** that characterize transcriptomic states.

This section visualizes the biological knowledge graph that grounds the model's communication pathway features.

In [ ]:
# --- Fig 29: Ligand-Receptor Communication Network (2-panel) ---
fig_lr, axes = plt.subplots(1, 2, figsize=(18, 8))

# Panel A: Bipartite LR network colored by signaling family
ax = axes[0]
# Organize by family
families = sorted(set(p.family for p in LUNG_LR_PRIORS))
ligands = sorted(set(p.ligand for p in LUNG_LR_PRIORS))
receptors = sorted(set(p.receptor for p in LUNG_LR_PRIORS))

# Position ligands on left, receptors on right
y_lig = {lig: i for i, lig in enumerate(ligands)}
y_rec = {rec: i for i, rec in enumerate(receptors)}
x_lig, x_rec = 0.0, 3.0

# Draw edges
for prior in LUNG_LR_PRIORS:
    color = LR_FAMILY_COLORS.get(prior.family, "#999")
    ax.plot([x_lig + 0.6, x_rec - 0.6],
            [y_lig[prior.ligand], y_rec[prior.receptor]],
            color=color, alpha=0.5 + 0.3 * prior.support, linewidth=1 + 1.5 * prior.support)

# Draw nodes
for lig, y in y_lig.items():
    ax.scatter(x_lig, y, s=100, c="#1f77b4", zorder=5, edgecolors="black", linewidths=0.8)
    ax.text(x_lig - 0.15, y, lig, ha="right", va="center", fontsize=8, fontweight="bold")
for rec, y in y_rec.items():
    ax.scatter(x_rec, y, s=100, c="#d62728", zorder=5, edgecolors="black", linewidths=0.8)
    ax.text(x_rec + 0.15, y, rec, ha="left", va="center", fontsize=8, fontweight="bold")

# Legend for families
legend_handles = [Line2D([0], [0], color=LR_FAMILY_COLORS[f], lw=2.5, label=f)
                  for f in families]
ax.legend(handles=legend_handles, title="Family", fontsize=7, title_fontsize=8,
          loc="upper center", ncol=3, frameon=True)
ax.set_xlim(-1.5, 4.5)
ax.set_ylim(-1, max(len(ligands), len(receptors)))
ax.set_title("A. Curated Ligand-Receptor Priors (24 pairs)", fontweight="bold", fontsize=11)
ax.text(x_lig, -0.8, "Ligands", ha="center", fontsize=10, fontweight="bold", color="#1f77b4")
ax.text(x_rec, -0.8, "Receptors", ha="center", fontsize=10, fontweight="bold", color="#d62728")
ax.axis("off")

# Panel B: Receiver program heatmap (programs × marker genes)
ax = axes[1]
all_genes = sorted(set(g for genes in RECEIVER_PROGRAMS.values() for g in genes))
prog_names = list(RECEIVER_PROGRAMS.keys())
matrix = np.zeros((len(prog_names), len(all_genes)))
for i, prog in enumerate(prog_names):
    for gene in RECEIVER_PROGRAMS[prog]:
        if gene in all_genes:
            matrix[i, all_genes.index(gene)] = 1.0

im = ax.imshow(matrix, aspect="auto", cmap="YlGn", interpolation="nearest")
ax.set_xticks(range(len(all_genes)))
ax.set_xticklabels(all_genes, fontsize=7, rotation=45, ha="right")
ax.set_yticks(range(len(prog_names)))
ax.set_yticklabels([p.replace("_", " ").title() for p in prog_names], fontsize=9)
ax.set_title("B. Receiver Programs (6 transcriptomic states)", fontweight="bold", fontsize=11)
ax.set_xlabel("Marker genes")

# Add family-to-program connections as text
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(range(len(prog_names)))
mapped_families = []
for prog in prog_names:
    fams = [f for f, p in FAMILY_TO_PROGRAM.items() if p == prog]
    mapped_families.append(", ".join(fams) if fams else "—")
ax2.set_yticklabels(mapped_families, fontsize=7, color="#555")
ax2.set_ylabel("Mapped L-R families", fontsize=9, color="#555")

fig_lr.suptitle("EA-MIST Biological Grounding: Communication Priors and Receiver Programs",
                fontsize=14, fontweight="bold")
fig_lr.tight_layout(rect=[0, 0, 1, 0.94])
fig_lr.savefig(FIGURE_ROOT / "fig29_lr_communication_network.png", dpi=300, bbox_inches="tight")
fig_lr.savefig(FIGURE_ROOT / "fig29_lr_communication_network.pdf", bbox_inches="tight")
display(fig_lr); plt.close(fig_lr)
print("✓ fig29_lr_communication_network.png/pdf")

# Print summary table
display(Markdown("### Communication Prior Summary"))
prior_df = pd.DataFrame([
    {"Ligand": p.ligand, "Receptor": p.receptor, "Family": p.family, "Support": p.support}
    for p in LUNG_LR_PRIORS
])
display(prior_df.style.background_gradient(subset=["Support"], cmap="YlOrRd")
        .format({"Support": "{:.2f}"}))

## Part VI: Atlas Ablation Benchmark

The rescue ablation evaluates **3 model families × 5 atlas configurations** under grouped ordinal 3-class labels with donor-held-out 3-fold CV and 50 HPO trials per fold.

### Ablation grid

| Model | Architecture | Complexity |
|-------|-------------|-----------|
| `pooled` | Mean-pool aggregation | Baseline |
| `deep_sets` | DeepSets φ→ρ MLP | Mid |
| `eamist` | Set transformer + prototypes | Full |

| Atlas mode | HLCA | LuCA | Contrast |
|-----------|------|------|---------|
| `no_atlas` | ✗ | ✗ | ✗ |
| `hlca_only` | ✓ | ✗ | ✗ |
| `luca_only` | ✗ | ✓ | ✗ |
| `hlca_luca` | ✓ | ✓ | ✗ |
| `hlca_luca_contrast` | ✓ | ✓ | ✓ |

### Composite selection score (grouped)
$$\text{score} = 0.40 \cdot \max(\rho_s, 0) + 0.30 \cdot \max(\kappa_w, 0) + 0.20 \cdot \text{bal\_acc} + 0.10 \cdot F_1^{macro}$$

In [ ]:
# --- Run or Load Ablation Benchmark ---
# Option A: Run the full benchmark (hours on GPU)
# benchmark_output = run_step("train_lesion", cfg)

# Option B: Load existing benchmark results from a completed run
import glob

BENCHMARK_ROOT = OUTPUT_ROOT / "rescue_ablation_20250608_v2" / "eamist_benchmark"
# Fallback: find any available benchmark directory
if not BENCHMARK_ROOT.exists():
    candidates = sorted(glob.glob(str(OUTPUT_ROOT / "*" / "eamist_benchmark")))
    if candidates:
        BENCHMARK_ROOT = Path(candidates[-1])
        print(f"Using benchmark at: {BENCHMARK_ROOT}")
    else:
        BENCHMARK_ROOT = None
        print("No benchmark results found. Run the ablation first:")
        print("  bash scripts/run_rescue_ablation.sh")

# Parse all fold results into a unified DataFrame
if BENCHMARK_ROOT and BENCHMARK_ROOT.exists():
    rows = []
    for metrics_file in sorted(BENCHMARK_ROOT.rglob("metrics.json")):
        parts = metrics_file.relative_to(BENCHMARK_ROOT).parts
        # Expected: reference_mode / model_family / fold_XX / seed_XXX / metrics.json
        if len(parts) >= 4:
            ref_mode, model_family, fold_dir, seed_dir = parts[0], parts[1], parts[2], parts[3]
            with open(metrics_file) as f:
                m = json.load(f)
            m["reference_mode"] = ref_mode
            m["model_family"] = model_family
            m["fold"] = fold_dir
            m["seed"] = seed_dir
            rows.append(m)

    if rows:
        results_df = pd.DataFrame(rows)
        print(f"Loaded {len(results_df)} result entries from {BENCHMARK_ROOT}")
        print(f"Models: {sorted(results_df['model_family'].unique())}")
        print(f"Modes:  {sorted(results_df['reference_mode'].unique())}")
        print(f"Folds:  {sorted(results_df['fold'].unique())}")
    else:
        results_df = pd.DataFrame()
        print("No metrics.json files found in benchmark directory.")
else:
    results_df = pd.DataFrame()

## Part VII: Results — Ablation Comparison and Metrics

### Key metrics
| Metric | Type | What it measures |
|--------|------|-----------------|
| `displacement_spearman` | Ordinal | Rank correlation of predicted progression displacement vs target |
| `grouped_weighted_kappa` | Ordinal | Linear-weighted Cohen's κ — penalizes distant misclassifications |
| `grouped_balanced_accuracy` | Classification | Mean per-class recall across the 3 grouped classes |
| `grouped_macro_f1` | Classification | Macro-averaged F1 |
| `composite_score` | Combined | 40% Spearman + 30% κ + 20% bal_acc + 10% F1 |

In [ ]:
# --- Ablation Comparison Table ---
if len(results_df) > 0:
    # Key metrics columns
    metric_cols = [
        "grouped_macro_f1", "grouped_balanced_accuracy", "grouped_weighted_kappa",
        "displacement_spearman", "displacement_mae", "composite_score",
    ]
    available_metrics = [c for c in metric_cols if c in results_df.columns]

    # Aggregate: mean ± std across folds and seeds
    agg_df = (
        results_df
        .groupby(["model_family", "reference_mode"])[available_metrics]
        .agg(["mean", "std"])
    )
    # Flatten multi-level columns
    agg_df.columns = [f"{m}_{s}" for m, s in agg_df.columns]
    agg_df = agg_df.reset_index()

    # Sort by composite score (descending)
    sort_col = "composite_score_mean" if "composite_score_mean" in agg_df.columns else available_metrics[0] + "_mean"
    agg_df = agg_df.sort_values(sort_col, ascending=False)

    display(Markdown("### Model × Atlas Mode Ablation (mean ± std across folds/seeds)"))
    display(agg_df.round(3))

    # Heatmap: composite score by model × mode
    if "composite_score_mean" in agg_df.columns:
        pivot = agg_df.pivot(index="model_family", columns="reference_mode", values="composite_score_mean")
        mode_order = ["no_atlas", "hlca_only", "luca_only", "hlca_luca", "hlca_luca_contrast"]
        pivot = pivot.reindex(columns=[c for c in mode_order if c in pivot.columns])

        fig, ax = plt.subplots(figsize=(10, 4))
        im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=10,
                            color="white" if val > pivot.values[~np.isnan(pivot.values)].mean() else "black")
        ax.set_title("Composite Selection Score (grouped)")
        plt.colorbar(im, ax=ax, label="Score")
        plt.tight_layout()
        plt.show()

    # Delta from no_atlas baseline
    if "no_atlas" in agg_df["reference_mode"].values and "composite_score_mean" in agg_df.columns:
        baseline = agg_df[agg_df["reference_mode"] == "no_atlas"].set_index("model_family")["composite_score_mean"]
        display(Markdown("### Atlas Lift (Δ composite score vs no_atlas)"))
        for _, row in agg_df.iterrows():
            bl = baseline.get(row["model_family"], np.nan)
            delta = row["composite_score_mean"] - bl
            if row["reference_mode"] != "no_atlas":
                print(f"  {row['model_family']:20s} {row['reference_mode']:25s}  Δ = {delta:+.3f}")
else:
    print("No results loaded. Run the ablation benchmark first.")

In [ ]:
# --- Confusion Matrices (Raw + Normalized) ---
if len(results_df) > 0 and BENCHMARK_ROOT:

    best_config = agg_df.iloc[0]
    best_model = best_config["model_family"]
    best_mode = best_config["reference_mode"]

    cm_files = sorted(BENCHMARK_ROOT.rglob(f"{best_mode}/{best_model}/*/confusion_matrix.json"))

    if cm_files:
        n_folds = min(len(cm_files), 3)
        n_classes = len(GROUPED_STAGE_ORDER)
        short_labels = ["Early", "Interm.", "Invasive"]

        # ── Raw counts (top row) + Normalized (bottom row) ──
        fig, axes = plt.subplots(2, n_folds, figsize=(5.5 * n_folds, 10))
        if n_folds == 1:
            axes = axes.reshape(2, 1)

        aggregated_cm = np.zeros((n_classes, n_classes), dtype=int)

        for idx, cm_file in enumerate(cm_files[:n_folds]):
            with open(cm_file) as f:
                cm_data = json.load(f)

            cm = np.zeros((n_classes, n_classes), dtype=int)
            for i in range(n_classes):
                for j in range(n_classes):
                    key = f"pred_{j}_true_{i}"
                    cm[i, j] = cm_data.get(key, 0)
            aggregated_cm += cm

            fold_name = cm_file.parent.parent.name

            # Raw counts
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0, idx],
                       xticklabels=short_labels, yticklabels=short_labels,
                       linewidths=1, linecolor="white", cbar=False,
                       annot_kws={"fontsize": 14, "fontweight": "bold"})
            axes[0, idx].set_xlabel("Predicted", fontsize=10)
            axes[0, idx].set_ylabel("True", fontsize=10)
            axes[0, idx].set_title(f"{fold_name} (raw)", fontsize=11, fontweight="bold")

            # Normalized (recall)
            cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
            sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[1, idx],
                       xticklabels=short_labels, yticklabels=short_labels,
                       linewidths=1, linecolor="white", cbar=False, vmin=0, vmax=1,
                       annot_kws={"fontsize": 14, "fontweight": "bold"})
            axes[1, idx].set_xlabel("Predicted", fontsize=10)
            axes[1, idx].set_ylabel("True", fontsize=10)
            axes[1, idx].set_title(f"{fold_name} (recall-normalized)", fontsize=11, fontweight="bold")

        fig.suptitle(f"Confusion Matrices — {best_model} / {best_mode}",
                    fontsize=14, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        fig.savefig(FIGURE_ROOT / "fig_confusion_matrices.png", dpi=300, bbox_inches="tight")
        display(fig); plt.close(fig)

        # ── Aggregated confusion matrix across all folds ──
        fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

        sns.heatmap(aggregated_cm, annot=True, fmt="d", cmap="Blues", ax=ax1,
                   xticklabels=short_labels, yticklabels=short_labels,
                   linewidths=1.5, linecolor="white",
                   annot_kws={"fontsize": 16, "fontweight": "bold"})
        ax1.set_xlabel("Predicted", fontsize=12); ax1.set_ylabel("True", fontsize=12)
        ax1.set_title("Aggregated (all folds, raw)", fontsize=12, fontweight="bold")

        agg_norm = aggregated_cm.astype(float) / (aggregated_cm.sum(axis=1, keepdims=True) + 1e-8)
        sns.heatmap(agg_norm, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax2,
                   xticklabels=short_labels, yticklabels=short_labels,
                   linewidths=1.5, linecolor="white", vmin=0, vmax=1,
                   annot_kws={"fontsize": 16, "fontweight": "bold"})
        ax2.set_xlabel("Predicted", fontsize=12); ax2.set_ylabel("True", fontsize=12)
        ax2.set_title("Aggregated (recall-normalized)", fontsize=12, fontweight="bold")

        fig2.suptitle(f"Aggregated Confusion — {best_model} / {best_mode}",
                     fontsize=14, fontweight="bold")
        fig2.tight_layout(rect=[0, 0, 1, 0.94])
        fig2.savefig(FIGURE_ROOT / "fig_confusion_aggregated.png", dpi=300, bbox_inches="tight")
        display(fig2); plt.close(fig2)

        # Per-class recall summary
        diag = np.diag(agg_norm)
        for i, (lbl, rec) in enumerate(zip(short_labels, diag)):
            print(f"  {lbl:>10s}: recall = {rec:.3f}  ({np.diag(aggregated_cm)[i]}/{aggregated_cm.sum(axis=1)[i]})")
    else:
        print(f"No confusion matrices found for {best_model}/{best_mode}")
else:
    print("No results to display.")

In [ ]:
# --- Displacement Analysis (Enhanced) ---
if len(results_df) > 0:
    disp_cols = ["displacement_spearman", "displacement_mae", "displacement_stage_monotonicity"]
    available_disp = [c for c in disp_cols if c in results_df.columns]

    if available_disp:
        # ── 1. Violin + strip per model family ──
        fig, axes = plt.subplots(1, len(available_disp), figsize=(6 * len(available_disp), 5))
        if not hasattr(axes, "__iter__"):
            axes = [axes]
        for ax, metric in zip(axes, available_disp):
            sns.violinplot(data=results_df, x="model_family", y=metric, ax=ax,
                          palette=MODEL_COLORS, inner=None, alpha=0.4, cut=0,
                          order=sorted(results_df["model_family"].unique()))
            sns.stripplot(data=results_df, x="model_family", y=metric, ax=ax,
                         hue="reference_mode", palette="Set2", size=5, alpha=0.8,
                         dodge=True, jitter=0.08, legend=metric == available_disp[-1],
                         order=sorted(results_df["model_family"].unique()))
            ax.set_title(metric.replace("_", " ").title(), fontsize=12, fontweight="bold")
            ax.set_xlabel("")
            ax.grid(axis="y", alpha=0.3)
            if metric == available_disp[-1]:
                ax.legend(title="Atlas Mode", fontsize=7, title_fontsize=8,
                         bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.suptitle("Displacement Metrics — Violin + Strip by Model × Atlas Mode",
                    fontsize=14, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 0.88, 0.94])
        fig.savefig(FIGURE_ROOT / "fig_displacement_violins.png", dpi=300, bbox_inches="tight")
        display(fig); plt.close(fig)

        # ── 2. Paired comparison: no_atlas vs hlca_luca per model ──
        if "displacement_spearman" in results_df.columns:
            paired_modes = ["no_atlas", "hlca_luca"]
            paired_data = results_df[results_df["reference_mode"].isin(paired_modes)]
            if len(paired_data) > 0:
                fig2, ax = plt.subplots(figsize=(8, 5))
                for model in sorted(paired_data["model_family"].unique()):
                    for fold in sorted(paired_data["fold"].unique()):
                        vals = {}
                        for mode in paired_modes:
                            v = paired_data[
                                (paired_data["model_family"] == model) &
                                (paired_data["fold"] == fold) &
                                (paired_data["reference_mode"] == mode)
                            ]["displacement_spearman"]
                            if len(v) > 0:
                                vals[mode] = v.mean()
                        if len(vals) == 2:
                            ax.plot([0, 1], [vals["no_atlas"], vals["hlca_luca"]],
                                   "o-", color=MODEL_COLORS.get(model, "gray"),
                                   alpha=0.6, markersize=6)
                # Add legend manually
                from matplotlib.lines import Line2D
                handles = [Line2D([0], [0], color=MODEL_COLORS[m], lw=2, label=m)
                          for m in sorted(paired_data["model_family"].unique()) if m in MODEL_COLORS]
                ax.legend(handles=handles, fontsize=9)
                ax.set_xticks([0, 1])
                ax.set_xticklabels(["no_atlas", "hlca_luca"], fontsize=12)
                ax.set_ylabel("Displacement Spearman ρ", fontsize=12)
                ax.set_title("Atlas Impact on Ordinal Displacement (Paired by Fold)",
                            fontsize=12, fontweight="bold")
                ax.grid(axis="y", alpha=0.3)
                fig2.tight_layout()
                fig2.savefig(FIGURE_ROOT / "fig_displacement_paired.png", dpi=300, bbox_inches="tight")
                display(fig2); plt.close(fig2)

        print("✓ Displacement analysis with violins and paired comparison rendered.")
    else:
        print("No displacement metrics found in results.")
else:
    print("No results to display.")

### Multi-Metric Model Comparison

Spider/radar charts and parallel coordinates reveal different aspects of each model × atlas configuration:
- **Radar chart**: Holistic comparison across all metrics — configurations with larger area are uniformly better
- **Parallel coordinates**: Trace each configuration across metrics to identify trade-offs and crossovers
- **Ridge distributions**: Per-metric distributions across folds/seeds show variability and robustness

In [ ]:
# --- Radar Charts, Parallel Coordinates, and Ridge Distributions ---
if len(results_df) > 0:
    metric_cols = [
        "grouped_macro_f1", "grouped_balanced_accuracy", "grouped_weighted_kappa",
        "displacement_spearman", "displacement_mae",
    ]
    available_metrics = [c for c in metric_cols if c in results_df.columns]

    if len(available_metrics) >= 3:
        # Build aggregated summary for radar/parallel coordinates
        radar_df = (
            results_df.groupby(["model_family", "reference_mode"])[available_metrics]
            .mean().reset_index()
        )
        radar_df["label"] = radar_df["model_family"] + " / " + radar_df["reference_mode"]

        # ── 1. Radar chart — top configurations ──
        # Select best mode per model family + overall best
        top_configs = (
            radar_df.sort_values(available_metrics[0], ascending=False)
            .drop_duplicates("model_family")
            .head(6)
        )
        fig_radar = plot_radar_chart(
            top_configs, available_metrics, labels_col="label",
            title="Multi-Metric Comparison (Best Mode per Model)",
            output_path=FIGURE_ROOT / "fig_radar_model_comparison.png",
        )
        display(fig_radar); plt.close(fig_radar)

        # ── 2. Parallel coordinates — all configurations ──
        fig_pc = plot_parallel_coordinates(
            radar_df, available_metrics, labels_col="label",
            title="Parallel Coordinates — All Model × Atlas Configurations",
            output_path=FIGURE_ROOT / "fig_parallel_coordinates.png",
        )
        display(fig_pc); plt.close(fig_pc)

        # ── 3. Ridge distributions (per metric, all configs pooled) ──
        ridge_data = {}
        for metric in available_metrics:
            vals = results_df[metric].dropna().values
            if len(vals) > 0:
                ridge_data[metric.replace("_", " ").title()] = vals

        if ridge_data:
            fig_ridge = plot_ridge_distributions(
                ridge_data,
                title="Metric Distributions across Folds and Seeds",
                output_path=FIGURE_ROOT / "fig_metric_ridge_distributions.png",
            )
            display(fig_ridge); plt.close(fig_ridge)

        # ── 4. Per-model-family metric distributions (violin + strip) ──
        fig_violin, axes = plt.subplots(1, len(available_metrics), figsize=(5 * len(available_metrics), 5))
        if not hasattr(axes, "__iter__"):
            axes = [axes]
        for ax, metric in zip(axes, available_metrics):
            sns.violinplot(data=results_df, x="model_family", y=metric, ax=ax,
                          palette=MODEL_COLORS, inner=None, alpha=0.4, cut=0)
            sns.stripplot(data=results_df, x="model_family", y=metric, ax=ax,
                         palette=MODEL_COLORS, size=4, alpha=0.8, jitter=0.15)
            ax.set_title(metric.replace("_", " ").title(), fontsize=11, fontweight="bold")
            ax.set_xlabel("")
            ax.grid(axis="y", alpha=0.3)
        fig_violin.suptitle("Per-Model Metric Distributions (all atlas modes)",
                           fontsize=13, fontweight="bold")
        fig_violin.tight_layout(rect=[0, 0, 1, 0.94])
        fig_violin.savefig(FIGURE_ROOT / "fig_model_violins.png", dpi=300, bbox_inches="tight")
        display(fig_violin); plt.close(fig_violin)

        # ── 5. Heatmap of mean ± std with proper annotation ──
        pivot_mean = radar_df.pivot(index="model_family", columns="reference_mode",
                                    values=available_metrics[0])
        mode_order = ["no_atlas", "hlca_only", "luca_only", "hlca_luca", "hlca_luca_contrast"]
        pivot_mean = pivot_mean.reindex(columns=[c for c in mode_order if c in pivot_mean.columns])

        # Get corresponding std values
        radar_std_df = (
            results_df.groupby(["model_family", "reference_mode"])[available_metrics]
            .std().reset_index()
        )
        pivot_std = radar_std_df.pivot(index="model_family", columns="reference_mode",
                                       values=available_metrics[0])
        pivot_std = pivot_std.reindex(columns=[c for c in mode_order if c in pivot_std.columns])

        fig_hm, ax = plt.subplots(figsize=(11, 5))
        sns.heatmap(pivot_mean, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
                   linewidths=1, linecolor="white", cbar_kws={"label": available_metrics[0]})
        # Overlay std as smaller text
        for i in range(len(pivot_mean.index)):
            for j in range(len(pivot_mean.columns)):
                std_val = pivot_std.iloc[i, j]
                if not np.isnan(std_val):
                    ax.text(j + 0.5, i + 0.72, f"±{std_val:.3f}", ha="center", va="center",
                            fontsize=7, color="gray", style="italic")
        ax.set_title(f"Model × Atlas Mode: {available_metrics[0]} (mean ± std)",
                    fontsize=12, fontweight="bold")
        fig_hm.tight_layout()
        fig_hm.savefig(FIGURE_ROOT / "fig_metric_heatmap_annotated.png", dpi=300, bbox_inches="tight")
        display(fig_hm); plt.close(fig_hm)

        print("✓ Radar, parallel coordinates, ridge, violin, and annotated heatmap rendered.")
    else:
        print("Insufficient metrics for advanced comparison plots.")
else:
    print("No results loaded.")

In [ ]:
# --- Negative Controls Comparison ---
# Load negative control results if available (run with --with-controls flag)
if BENCHMARK_ROOT:
    control_dir = BENCHMARK_ROOT.parent / "negative_controls"
    control_rows = []
    for metrics_file in sorted((control_dir).rglob("metrics.json")) if control_dir.exists() else []:
        parts = metrics_file.relative_to(control_dir).parts
        if len(parts) >= 4:
            control_type, model_family, fold_dir, seed_dir = parts[0], parts[1], parts[2], parts[3]
            with open(metrics_file) as f:
                m = json.load(f)
            m["control_type"] = control_type
            m["model_family"] = model_family
            m["fold"] = fold_dir
            m["seed"] = seed_dir
            control_rows.append(m)

    if control_rows:
        controls_df = pd.DataFrame(control_rows)
        display(Markdown("### Negative Controls"))
        display(Markdown("Atlas label shuffle should produce lower scores than intact `hlca_luca`."))

        control_agg = (
            controls_df
            .groupby(["control_type", "model_family"])
            [["composite_score", "grouped_balanced_accuracy", "displacement_spearman"]]
            .agg(["mean", "std"])
        )
        control_agg.columns = [f"{m}_{s}" for m, s in control_agg.columns]
        control_agg = control_agg.reset_index()
        display(control_agg.round(3))

        # Compare intact vs shuffled
        if len(results_df) > 0:
            intact = results_df[results_df["reference_mode"] == "hlca_luca"]
            if len(intact) > 0 and "composite_score" in intact.columns:
                intact_score = intact.groupby("model_family")["composite_score"].mean()
                shuffle_df = controls_df[controls_df["control_type"] == "atlas_label_shuffle"]
                if len(shuffle_df) > 0 and "composite_score" in shuffle_df.columns:
                    shuffle_score = shuffle_df.groupby("model_family")["composite_score"].mean()
                    display(Markdown("### Atlas Shuffle Impact (Δ = intact − shuffled)"))
                    for model in sorted(set(intact_score.index) & set(shuffle_score.index)):
                        delta = intact_score[model] - shuffle_score[model]
                        print(f"  {model:20s}  Δ = {delta:+.3f}  ({'atlas signal confirmed' if delta > 0.05 else 'weak signal'})")
    else:
        print("No negative control results found. Run with: bash scripts/run_rescue_ablation.sh --with-controls")
else:
    print("No benchmark root to check for controls.")

## Part VIII: Transcriptomic, Cell-Level, and Feature Structure Analysis

This section examines the biological structure that the model leverages:
- **Cell-type composition** by stage — Which cell types dominate at each progression point?
- **Atlas similarity profiles** — How do HLCA (healthy) and LuCA (cancer) features shift across stages?
- **Hierarchical clustermaps** — Discover which cell-type similarities co-cluster
- **Atlas divergence** — Scatter with marginal distributions showing healthy↔cancer reference trade-off
- **Effect sizes** — Top discriminative atlas features for each group
- **Niche heterogeneity** — Within-lesion diversity of neighborhood phenotypes
- **Cross-atlas correlation** — HLCA × LuCA correlation block and clustered heatmap

In [ ]:
# --- Cell-Type Composition, Atlas Feature Profiles, and Clustermaps ---
if bags_path.exists():
    bags_df = pd.read_parquet(bags_path)
    bags_df["grouped_label"] = bags_df["stage"].map(STAGE_TO_GROUP)

    hlca_cols = sorted([c for c in bags_df.columns if c.startswith("hlca_")])
    luca_cols = sorted([c for c in bags_df.columns if c.startswith("luca_")])

    if hlca_cols and luca_cols:
        # ── 1. Heatmaps: atlas profiles by canonical stage ──
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        hlca_by_stage = bags_df.groupby("stage")[hlca_cols].mean().reindex(CANONICAL_STAGE_ORDER)
        im0 = axes[0].imshow(hlca_by_stage.values, aspect="auto", cmap="YlGnBu")
        axes[0].set_yticks(range(len(CANONICAL_STAGE_ORDER)))
        axes[0].set_yticklabels(CANONICAL_STAGE_ORDER)
        axes[0].set_xticks(range(len(hlca_cols)))
        axes[0].set_xticklabels([c.replace("hlca_", "") for c in hlca_cols], rotation=90, fontsize=7)
        axes[0].set_title("HLCA Similarity Profile by Stage", fontweight="bold")
        plt.colorbar(im0, ax=axes[0], label="Mean cosine sim.", shrink=0.8)

        luca_by_stage = bags_df.groupby("stage")[luca_cols].mean().reindex(CANONICAL_STAGE_ORDER)
        im1 = axes[1].imshow(luca_by_stage.values, aspect="auto", cmap="YlOrRd")
        axes[1].set_yticks(range(len(CANONICAL_STAGE_ORDER)))
        axes[1].set_yticklabels(CANONICAL_STAGE_ORDER)
        axes[1].set_xticks(range(len(luca_cols)))
        axes[1].set_xticklabels([c.replace("luca_", "") for c in luca_cols], rotation=90, fontsize=7)
        axes[1].set_title("LuCA Similarity Profile by Stage", fontweight="bold")
        plt.colorbar(im1, ax=axes[1], label="Mean cosine sim.", shrink=0.8)
        fig.suptitle("Atlas Feature Profiles across Disease Stages", fontsize=14, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        fig.savefig(FIGURE_ROOT / "fig_atlas_profiles_heatmap.png", dpi=300, bbox_inches="tight")
        display(fig); plt.close(fig)

        # ── 2. Seaborn clustermap with hierarchical clustering (HLCA) ──
        hlca_cluster_data = bags_df.groupby("stage")[hlca_cols].mean().reindex(CANONICAL_STAGE_ORDER)
        hlca_cluster_data.columns = [c.replace("hlca_", "") for c in hlca_cols]
        g1 = sns.clustermap(
            hlca_cluster_data, cmap="YlGnBu", figsize=(10, 5), linewidths=0.5,
            row_cluster=False, col_cluster=True,  # cluster cell types, keep stage order
            standard_scale=1,  # z-score columns
            cbar_kws={"label": "Z-score (column)"},
            dendrogram_ratio=(0.08, 0.15),
        )
        g1.figure.suptitle("HLCA Features — Hierarchically Clustered Cell Types",
                        fontsize=13, fontweight="bold", y=1.02)
        g1.savefig(FIGURE_ROOT / "fig_hlca_clustermap.png", dpi=300, bbox_inches="tight")
        display(g1.figure); plt.close(g1.figure)

        # ── 3. Seaborn clustermap (LuCA) ──
        luca_cluster_data = bags_df.groupby("stage")[luca_cols].mean().reindex(CANONICAL_STAGE_ORDER)
        luca_cluster_data.columns = [c.replace("luca_", "") for c in luca_cols]
        g2 = sns.clustermap(
            luca_cluster_data, cmap="YlOrRd", figsize=(12, 5), linewidths=0.5,
            row_cluster=False, col_cluster=True,
            standard_scale=1,
            cbar_kws={"label": "Z-score (column)"},
            dendrogram_ratio=(0.08, 0.15),
        )
        g2.figure.suptitle("LuCA Features — Hierarchically Clustered Cell Types",
                        fontsize=13, fontweight="bold", y=1.02)
        g2.savefig(FIGURE_ROOT / "fig_luca_clustermap.png", dpi=300, bbox_inches="tight")
        display(g2.figure); plt.close(g2.figure)

        # ── 4. Atlas divergence scatter with marginal distributions ──
        fig, ax = plt.subplots(figsize=(8, 7))

        # Use JointGrid for marginal histograms
        sample_n = min(3000, len(bags_df))
        sample_idx = np.random.default_rng(42).choice(len(bags_df), sample_n, replace=False)
        sample = bags_df.iloc[sample_idx]
        sample["hlca_mean"] = sample[hlca_cols].mean(axis=1)
        sample["luca_mean"] = sample[luca_cols].mean(axis=1)

        jg = sns.JointGrid(data=sample, x="hlca_mean", y="luca_mean", hue="grouped_label",
                           hue_order=GROUPED_STAGE_ORDER, palette=GROUP_COLORS, height=7)
        jg.plot_joint(sns.scatterplot, s=8, alpha=0.4, linewidth=0, rasterized=True)
        jg.plot_marginals(sns.kdeplot, fill=True, alpha=0.3, common_norm=False, bw_adjust=1.2)
        jg.set_axis_labels("Mean HLCA Sim. (healthy reference)", "Mean LuCA Sim. (cancer reference)")
        jg.figure.suptitle("Atlas Divergence with Marginal Densities",
                          fontsize=13, fontweight="bold", y=1.02)
        jg.savefig(FIGURE_ROOT / "fig_atlas_divergence_joint.png", dpi=300, bbox_inches="tight")
        display(jg.figure); plt.close(jg.figure)

        # ── 5. Stage-specific top discriminative features ──
        atlas_cols_all = hlca_cols + luca_cols
        grouped_means = bags_df.groupby("grouped_label")[atlas_cols_all].mean()
        # Effect size: difference from grand mean normalized by pooled std
        grand_mean = bags_df[atlas_cols_all].mean()
        pooled_std = bags_df[atlas_cols_all].std()
        effect_sizes = (grouped_means - grand_mean) / (pooled_std + 1e-8)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
        for ax, grp in zip(axes, GROUPED_STAGE_ORDER):
            es = effect_sizes.loc[grp].sort_values()
            top_neg = es.head(5)
            top_pos = es.tail(5)
            combined = pd.concat([top_neg, top_pos])
            colors = ["#2166AC" if v < 0 else "#B2182B" for v in combined.values]
            ax.barh(range(len(combined)), combined.values, color=colors, edgecolor="white")
            labels = [c.replace("hlca_", "H:").replace("luca_", "L:") for c in combined.index]
            ax.set_yticks(range(len(combined)))
            ax.set_yticklabels(labels, fontsize=9)
            ax.set_xlabel("Effect size (Cohen's d)", fontsize=10)
            ax.set_title(f"{grp}", fontsize=11, fontweight="bold")
            ax.axvline(x=0, color="black", linewidth=0.8)
            ax.grid(axis="x", alpha=0.3)
        fig.suptitle("Top Discriminative Atlas Features by Group (vs Grand Mean)",
                     fontsize=13, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        fig.savefig(FIGURE_ROOT / "fig_atlas_effect_sizes.png", dpi=300, bbox_inches="tight")
        display(fig); plt.close(fig)

        print("✓ Atlas profiles, clustermaps, divergence scatter, and effect sizes rendered.")
    else:
        print("No HLCA/LuCA columns found.")
else:
    print("Bags parquet not found.")

In [ ]:
# --- Within-Lesion Niche Heterogeneity (Enhanced) ---
if bags_path.exists():
    bags_df_het = pd.read_parquet(bags_path)
    hlca_cols_h = sorted([c for c in bags_df_het.columns if c.startswith("hlca_")])
    luca_cols_h = sorted([c for c in bags_df_het.columns if c.startswith("luca_")])
    atlas_cols_h = hlca_cols_h + luca_cols_h

    if atlas_cols_h:
        # Compute per-lesion feature variance
        lesion_stats = (
            bags_df_het.groupby(["lesion_id", "stage", "donor_id"])[atlas_cols_h]
            .agg(["mean", "std"])
        )
        lesion_stats.columns = [f"{col}_{stat}" for col, stat in lesion_stats.columns]
        lesion_stats = lesion_stats.reset_index()
        lesion_stats["grouped_label"] = lesion_stats["stage"].map(STAGE_TO_GROUP)

        std_cols = [c for c in lesion_stats.columns if c.endswith("_std")]
        lesion_stats["mean_atlas_std"] = lesion_stats[std_cols].mean(axis=1)

        nhood_counts = bags_df_het.groupby("lesion_id").size().rename("n_neighborhoods")
        merged = lesion_stats.merge(nhood_counts, on="lesion_id")

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # ── Panel 1: Heterogeneity violin by group ──
        het_data = []
        for grp in GROUPED_STAGE_ORDER:
            vals = lesion_stats[lesion_stats["grouped_label"] == grp]["mean_atlas_std"].values
            het_data.extend([(v, grp) for v in vals])
        het_df = pd.DataFrame(het_data, columns=["mean_atlas_std", "group"])
        sns.violinplot(data=het_df, x="group", y="mean_atlas_std", ax=axes[0],
                      palette=GROUP_COLORS, inner=None, alpha=0.4, cut=0,
                      order=GROUPED_STAGE_ORDER)
        sns.stripplot(data=het_df, x="group", y="mean_atlas_std", ax=axes[0],
                     palette=GROUP_COLORS, size=8, alpha=0.8, jitter=0.1,
                     order=GROUPED_STAGE_ORDER)
        axes[0].set_ylabel("Mean within-lesion atlas feature σ", fontsize=11)
        axes[0].set_title("Niche Heterogeneity by Group", fontsize=12, fontweight="bold")
        axes[0].grid(axis="y", alpha=0.3)
        axes[0].set_xlabel("")

        # ── Panel 2: Scatter heterogeneity vs size, with regression line ──
        for grp in GROUPED_STAGE_ORDER:
            sub = merged[merged["grouped_label"] == grp]
            axes[1].scatter(sub["n_neighborhoods"], sub["mean_atlas_std"],
                           color=GROUP_COLORS[grp], alpha=0.8, s=50, label=grp,
                           edgecolors="white", linewidths=0.8)
        # Add regression line
        from scipy.stats import spearmanr as _sp
        rho, pval = _sp(merged["n_neighborhoods"], merged["mean_atlas_std"])
        z = np.polyfit(merged["n_neighborhoods"], merged["mean_atlas_std"], 1)
        p = np.poly1d(z)
        x_line = np.linspace(merged["n_neighborhoods"].min(), merged["n_neighborhoods"].max(), 100)
        axes[1].plot(x_line, p(x_line), "--", color="gray", alpha=0.7, lw=1.5)
        axes[1].set_xlabel("Neighborhoods per lesion", fontsize=11)
        axes[1].set_ylabel("Mean atlas feature σ", fontsize=11)
        axes[1].set_title(f"Heterogeneity vs Size (ρ={rho:.2f}, p={pval:.3f})",
                         fontsize=12, fontweight="bold")
        axes[1].legend(fontsize=9)
        axes[1].grid(alpha=0.3)

        # ── Panel 3: Per-feature heterogeneity heatmap (group × feature) ──
        # Mean std per group and atlas feature
        for_hm = lesion_stats.groupby("grouped_label")[std_cols].mean()
        for_hm.columns = [c.replace("_std", "").replace("hlca_", "H:").replace("luca_", "L:") for c in std_cols]
        for_hm = for_hm.reindex(GROUPED_STAGE_ORDER)
        im = axes[2].imshow(for_hm.values, aspect="auto", cmap="viridis")
        axes[2].set_yticks(range(len(GROUPED_STAGE_ORDER)))
        axes[2].set_yticklabels(GROUPED_STAGE_ORDER)
        axes[2].set_xticks(range(len(for_hm.columns)))
        axes[2].set_xticklabels(for_hm.columns, rotation=90, fontsize=6)
        axes[2].set_title("Per-Feature Heterogeneity by Group", fontsize=12, fontweight="bold")
        plt.colorbar(im, ax=axes[2], label="Mean within-lesion σ", shrink=0.8)

        fig.suptitle("Niche Heterogeneity Analysis", fontsize=14, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        fig.savefig(FIGURE_ROOT / "fig_niche_heterogeneity.png", dpi=300, bbox_inches="tight")
        display(fig); plt.close(fig)

        # ── Cell-type receiver state distribution by stage ──
        if "receiver_state_id" in bags_df_het.columns:
            ct_by_stage = pd.crosstab(bags_df_het["stage"], bags_df_het["receiver_state_id"],
                                      normalize="index")
            ct_by_stage = ct_by_stage.reindex(CANONICAL_STAGE_ORDER)
            top_states = ct_by_stage.sum().nlargest(15).index
            ct_top = ct_by_stage[top_states]

            fig2, ax = plt.subplots(figsize=(14, 5))
            ct_top.plot.bar(stacked=True, ax=ax, colormap="tab20", width=0.8)
            ax.set_title("Receiver Cell-Type Distribution by Stage (Top 15)",
                        fontsize=13, fontweight="bold")
            ax.set_xlabel("Stage", fontsize=12); ax.set_ylabel("Fraction", fontsize=12)
            ax.legend(title="State ID", bbox_to_anchor=(1.02, 1), loc="upper left",
                     fontsize=7, ncol=2, title_fontsize=9)
            fig2.tight_layout()
            fig2.savefig(FIGURE_ROOT / "fig_receiver_distribution.png", dpi=300, bbox_inches="tight")
            display(fig2); plt.close(fig2)

        print("✓ Niche heterogeneity and receiver distribution analysis rendered.")
    else:
        print("No atlas columns available.")
else:
    print("Bags parquet not available.")

### Feature Correlation and Inter-Atlas Structure

Cross-correlation between atlas features reveals which cell-type similarities co-occur across neighborhoods.
Block structure in the correlation matrix indicates feature groups that may be redundantly encoded,
while anti-correlated features highlight biological trade-offs (e.g., healthy vs cancer niches).

In [ ]:
# --- Atlas Feature Correlation Matrix and Inter-Atlas Analysis ---
if bags_path.exists():
    bags_df = pd.read_parquet(bags_path)
    hlca_cols = sorted([c for c in bags_df.columns if c.startswith("hlca_")])
    luca_cols = sorted([c for c in bags_df.columns if c.startswith("luca_")])
    atlas_cols = hlca_cols + luca_cols

    if atlas_cols:
        # Subsample for efficiency
        n_corr = min(20000, len(bags_df))
        corr_sample = bags_df.sample(n_corr, random_state=42)

        # ── 1. Full atlas feature correlation matrix ──
        fig_corr = plot_correlation_matrix(
            corr_sample, metrics=atlas_cols,
            title="Atlas Feature Correlation (Spearman)",
            method="spearman",
            output_path=FIGURE_ROOT / "fig_atlas_correlation_matrix.png",
        )
        display(fig_corr); plt.close(fig_corr)

        # ── 2. Seaborn clustermap with both-axis clustering ──
        corr_mat = corr_sample[atlas_cols].corr(method="spearman")
        # Rename for readability
        short_names = [c.replace("hlca_", "H:").replace("luca_", "L:") for c in atlas_cols]
        corr_mat.index = short_names
        corr_mat.columns = short_names

        # Color sidebar: HLCA vs LuCA
        row_colors = pd.Series(
            ["#2166AC"] * len(hlca_cols) + ["#B2182B"] * len(luca_cols),
            index=short_names, name="Atlas"
        )

        g = sns.clustermap(
            corr_mat, cmap="RdBu_r", vmin=-1, vmax=1, figsize=(12, 11),
            linewidths=0.3, row_colors=row_colors, col_colors=row_colors,
            dendrogram_ratio=(0.12, 0.12),
            cbar_kws={"label": "Spearman ρ", "shrink": 0.6},
        )
        g.fig.suptitle("Hierarchically Clustered Atlas Feature Correlation",
                       fontsize=14, fontweight="bold", y=1.01)
        # Add legend for atlas colors
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor="#2166AC", label="HLCA (healthy)"),
                          Patch(facecolor="#B2182B", label="LuCA (cancer)")]
        g.ax_heatmap.legend(handles=legend_elements, loc="lower left",
                           fontsize=9, frameon=True, framealpha=0.9)
        g.savefig(FIGURE_ROOT / "fig_atlas_corr_clustermap.png", dpi=300, bbox_inches="tight")
        display(g.fig); plt.close(g.fig)

        # ── 3. Cross-atlas correlation block (HLCA rows × LuCA cols) ──
        cross_corr = corr_sample[hlca_cols].corrwith(
            corr_sample[luca_cols].rename(columns=dict(zip(luca_cols, hlca_cols))),
            method="spearman"
        )
        # Better: compute full cross-atlas block
        cross_block = corr_sample[hlca_cols + luca_cols].corr(method="spearman").loc[hlca_cols, luca_cols]
        cross_block.index = [c.replace("hlca_", "") for c in hlca_cols]
        cross_block.columns = [c.replace("luca_", "") for c in luca_cols]

        fig3, ax = plt.subplots(figsize=(12, 8))
        im = ax.imshow(cross_block.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(len(cross_block.columns)))
        ax.set_xticklabels(cross_block.columns, rotation=90, fontsize=8)
        ax.set_yticks(range(len(cross_block.index)))
        ax.set_yticklabels(cross_block.index, fontsize=8)
        for i in range(len(cross_block.index)):
            for j in range(len(cross_block.columns)):
                v = cross_block.values[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                        color="white" if abs(v) > 0.5 else "black")
        ax.set_xlabel("LuCA (cancer cell types)", fontsize=12, fontweight="bold")
        ax.set_ylabel("HLCA (healthy cell types)", fontsize=12, fontweight="bold")
        ax.set_title("Cross-Atlas Correlation: HLCA × LuCA", fontsize=13, fontweight="bold")
        plt.colorbar(im, ax=ax, label="Spearman ρ", shrink=0.7)
        fig3.tight_layout()
        fig3.savefig(FIGURE_ROOT / "fig_cross_atlas_correlation.png", dpi=300, bbox_inches="tight")
        display(fig3); plt.close(fig3)

        print("✓ Correlation matrix, clustered heatmap, and cross-atlas block rendered.")
    else:
        print("No atlas columns found.")
else:
    print("Bags parquet not found.")

## Part IX: Publication Figures and Results Summary

### Complete Figure Inventory

| Figure | Content | Panel Count | Method |
|--------|---------|-------------|--------|
| Fig 1 | Method overview schematic | 1 | `save_method_overview_figure` |
| Fig 2 | snRNA 4-embedding panel (PCA+%/UMAP/t-SNE/PHATE) | 4 | `plot_four_embeddings` |
| Fig 3 | PCA scree + cumulative variance | 2 | Inline |
| Fig 4 | UMAP with stage density contours | 1 | Inline + `gaussian_kde` |
| Fig 5 | Niche-level 4-embedding (grouped + canonical) | 8 | `plot_four_embeddings` |
| Fig 6 | UMAP with 95% confidence ellipses | 1 | Inline + `confidence_ellipse` |
| Fig 7 | Lesion-level 4-embedding with ellipses | 4 | Inline |
| Fig 8 | Annotated lesion UMAP | 1 | Inline |
| Fig 9 | Lesion 3D PCA | 1 | `plot_3d_embedding` |
| Fig 10 | Atlas profiles heatmap (HLCA + LuCA) | 2 | Inline |
| Fig 11 | HLCA/LuCA clustermaps with dendrograms | 2 | `sns.clustermap` |
| Fig 12 | Atlas divergence joint plot | 1 | `sns.JointGrid` |
| Fig 13 | Stage-specific effect sizes | 3 | Inline |
| Fig 14 | Ablation heatmap (annotated) | 1 | `sns.heatmap` |
| Fig 15 | Confusion matrices (raw + normalized, per-fold + aggregated) | 8 | `sns.heatmap` |
| Fig 16 | Displacement violins + paired comparison | 3 | `sns.violinplot` |
| Fig 17 | Radar chart (multi-metric) | 1 | `plot_radar_chart` |
| Fig 18 | Parallel coordinates | 1 | `plot_parallel_coordinates` |
| Fig 19 | Ridge distributions | 1 | `plot_ridge_distributions` |
| Fig 20 | Per-model metric violins | N | `sns.violinplot` |
| Fig 21 | Niche heterogeneity + receiver composition | 3 | Inline |
| Fig 22 | Cross-atlas correlation + clustermap | 3 | `plot_correlation_matrix` |
| Fig 23 | Composite multi-panel summary | 6 | Assembly |
| **Fig 24** | **EA-MIST architecture diagram with 7 token types** | **1** | **Inline (matplotlib)** |
| **Fig 25** | **Prototype bottleneck analysis (composition + PCA + occupancy)** | **3** | **Inline** |
| **Fig 26** | **Attention analysis (token-type importance + neighborhood importance)** | **2** | **Inline** |
| **Fig 27** | **Niche transition score distributions** | **2** | **Inline** |
| **Fig 28** | **Learned representations (embedding PCA + predictions + proto-stage)** | **3** | **Inline** |
| **Fig 29** | **Ligand-receptor communication network + receiver programs** | **2** | **Inline** |

### Key claims this notebook supports

1. **Atlas features carry stage signal** — `hlca_luca` outperforms `no_atlas` across all model families
2. **Both atlases contribute** — Neither `hlca_only` nor `luca_only` alone matches `hlca_luca`
3. **Ordinal structure is preserved** — High displacement Spearman rho and weighted kappa
4. **Niche-level separation is visible** — DR embeddings show group clustering before any model training
5. **Cross-atlas structure** — HLCA and LuCA features show informative correlation/anti-correlation patterns
6. **Negative controls confirm specificity** — Performance drops under atlas label shuffle
7. **Prototype motifs are diverse and stage-associated** — K=16 prototypes capture distinct niche patterns with differential occupancy across progression stages
8. **Token-type attention is biologically coherent** — The transformer learns to weight atlas and communication tokens appropriately
9. **Learned embeddings capture ordinal progression** — Lesion representations show smooth stage separation in PCA space
10. **Communication priors ground the model in LUAD biology** — 24 curated L-R pairs from 9 signaling families connect to 6 receiver transcriptomic programs

In [ ]:
# --- Publication Figures: Assembly and Export ---
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)

# ── Fig 1: Method overview ──
save_method_overview_figure(FIGURE_ROOT / "fig1_method_overview.png")
print("✓ fig1_method_overview.png")

# ── Composite Figure: Multi-panel summary (6 key panels) ──
if bags_path.exists() and len(results_df) > 0:
    fig_comp, axes = plt.subplots(2, 3, figsize=(20, 13))

    # Panel A: Lesion UMAP by grouped label
    if "UMAP" in lesion_emb:
        ax = axes[0, 0]
        umap_l = lesion_emb["UMAP"][0]
        for grp in GROUPED_STAGE_ORDER:
            mask = lesion_groups == grp
            ax.scatter(umap_l[mask, 0], umap_l[mask, 1], s=60, alpha=0.8,
                       color=GROUP_COLORS[grp], label=grp, edgecolors="white",
                       linewidths=0.8, zorder=3)
            confidence_ellipse(umap_l[mask, 0], umap_l[mask, 1], ax, n_std=2.0,
                               facecolor=GROUP_COLORS[grp], alpha=0.10,
                               edgecolor=GROUP_COLORS[grp], linewidth=2)
        ax.set_title("A. Lesion-Level UMAP", fontsize=12, fontweight="bold")
        ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
        ax.legend(fontsize=8, frameon=True)

    # Panel B: Composite score heatmap
    ax = axes[0, 1]
    if "composite_score_mean" in agg_df.columns:
        pivot = agg_df.pivot(index="model_family", columns="reference_mode",
                             values="composite_score_mean")
        mode_order = ["no_atlas", "hlca_only", "luca_only", "hlca_luca", "hlca_luca_contrast"]
        pivot = pivot.reindex(columns=[c for c in mode_order if c in pivot.columns])
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
                   linewidths=1, linecolor="white", cbar_kws={"shrink": 0.8})
        ax.set_title("B. Ablation: Composite Score", fontsize=12, fontweight="bold")
    else:
        ax.text(0.5, 0.5, "No composite scores", ha="center", va="center")

    # Panel C: Aggregated confusion matrix
    ax = axes[0, 2]
    if 'aggregated_cm' in dir():
        agg_norm = aggregated_cm.astype(float) / (aggregated_cm.sum(axis=1, keepdims=True) + 1e-8)
        sns.heatmap(agg_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax,
                   xticklabels=["Early", "Interm.", "Invasive"],
                   yticklabels=["Early", "Interm.", "Invasive"],
                   linewidths=1.5, linecolor="white", vmin=0, vmax=1,
                   annot_kws={"fontsize": 13, "fontweight": "bold"})
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title("C. Confusion (Aggregated)", fontsize=12, fontweight="bold")
    else:
        ax.text(0.5, 0.5, "No confusion data", ha="center", va="center")

    # Panel D: Atlas divergence scatter
    ax = axes[1, 0]
    if hlca_cols and luca_cols:
        _bags = pd.read_parquet(bags_path)
        _bags["grouped_label"] = _bags["stage"].map(STAGE_TO_GROUP)
        sample_comp = _bags.sample(min(3000, len(_bags)), random_state=42)
        for grp in GROUPED_STAGE_ORDER:
            sub = sample_comp[sample_comp["grouped_label"] == grp]
            ax.scatter(sub[hlca_cols].mean(axis=1), sub[luca_cols].mean(axis=1),
                       s=4, alpha=0.3, color=GROUP_COLORS[grp], label=grp, rasterized=True)
        ax.set_xlabel("Mean HLCA sim."); ax.set_ylabel("Mean LuCA sim.")
        ax.set_title("D. Atlas Divergence", fontsize=12, fontweight="bold")
        ax.legend(fontsize=8, markerscale=3)

    # Panel E: Displacement Spearman by model
    ax = axes[1, 1]
    if "displacement_spearman" in results_df.columns:
        sns.boxplot(data=results_df, x="model_family", y="displacement_spearman",
                   hue="reference_mode", ax=ax, palette="Set2", fliersize=3)
        ax.set_title("E. Displacement Spearman ρ", fontsize=12, fontweight="bold")
        ax.legend(fontsize=6, title="mode", title_fontsize=7)
        ax.set_xlabel("")
    else:
        ax.text(0.5, 0.5, "No displacement data", ha="center", va="center")

    # Panel F: PCA variance (niche atlas features)
    ax = axes[1, 2]
    if 'pca_niche' in dir():
        var_n = pca_niche.explained_variance_ratio_ * 100
        cum_n = np.cumsum(var_n)
        ax.bar(range(1, len(var_n)+1), var_n, color="#0E7490", edgecolor="white", alpha=0.7)
        ax2_twin = ax.twinx()
        ax2_twin.plot(range(1, len(cum_n)+1), cum_n, "o-", color="#D95F02", lw=2)
        ax2_twin.axhline(y=90, color="gray", ls="--", alpha=0.5)
        ax.set_xlabel("PC"); ax.set_ylabel("Var. Explained (%)")
        ax2_twin.set_ylabel("Cum. %")
        ax.set_title("F. Atlas PCA Variance", fontsize=12, fontweight="bold")
    else:
        ax.text(0.5, 0.5, "No PCA data", ha="center", va="center")

    fig_comp.suptitle("StageBridge — EA-MIST Rescue Ablation Summary",
                     fontsize=16, fontweight="bold")
    fig_comp.tight_layout(rect=[0, 0, 1, 0.96])
    fig_comp.savefig(FIGURE_ROOT / "fig_composite_summary.png", dpi=300, bbox_inches="tight")
    fig_comp.savefig(FIGURE_ROOT / "fig_composite_summary.pdf", bbox_inches="tight")
    display(fig_comp); plt.close(fig_comp)
    print("✓ fig_composite_summary.png/pdf")
else:
    print("Composite figure requires both bags data and benchmark results.")

# ── Export all generated figures inventory ──
all_figs = sorted(FIGURE_ROOT.glob("fig_*.png"))
display(Markdown(f"### Generated Figures: {len(all_figs)} files"))
for f in all_figs:
    sz = f.stat().st_size / 1024
    pdf_exists = "✓" if f.with_suffix(".pdf").exists() else "–"
    print(f"  {f.name:50s} {sz:7.0f} KB  PDF: {pdf_exists}")

# ── Results Summary Table ──
display(Markdown("---"))
display(Markdown("### Run Summary"))

summary_rows = []
if len(results_df) > 0 and "composite_score_mean" in agg_df.columns:
    best = agg_df.iloc[0]
    summary_rows.append(("Best configuration", f"{best['model_family']} / {best['reference_mode']}"))
    summary_rows.append(("Composite score", f"{best['composite_score_mean']:.3f} ± {best.get('composite_score_std', 0):.3f}"))
    for m in ["grouped_balanced_accuracy", "grouped_weighted_kappa", "displacement_spearman", "grouped_macro_f1"]:
        if f"{m}_mean" in best:
            summary_rows.append((m.replace("_", " ").title(),
                                f"{best[f'{m}_mean']:.3f} ± {best.get(f'{m}_std', 0):.3f}"))

summary_rows.extend([
    ("Dataset", "56 lesions, 25 donors, 639K neighborhoods"),
    ("Labels", "3-class grouped ordinal (early/intermediate/invasive)"),
    ("CV strategy", "Donor-held-out 3-fold"),
    ("HPO", "50 Optuna trials/fold"),
    ("Ablation grid", "3 models × 5 atlas modes = 15 configurations"),
    ("Figures generated", f"{len(all_figs)} PNG + PDF pairs"),
    ("DR methods", f"PCA ✓  UMAP {'✓' if HAS_UMAP else '✗'}  t-SNE ✓  PHATE {'✓' if HAS_PHATE else '✗'}"),
])

display(pd.DataFrame(summary_rows, columns=["Item", "Value"]))
print("\n✓ Pipeline complete.")